# 07 · Transfer status: a $10k census of swap rows and integrator strata

`GET https://li.quest/v1/status?txHash=<hash>` for two further sample designs over the swap rows of the data (`hasSourceSwaps = true`, the rows that carry LI.FI fee lines): a census of every swap row at $10,000 or more of headline USD (`census_swap_10k`), and per integrator string 30, 10 or every event by the integrator's swap-row count (`integrator_strata`). Hashes already in `lifi_api_transfer_status_sample_2600_raw.jsonl` or `lifi_api_transfer_status_edge_cases_raw.jsonl` are not refetched: their raw record is reused with its original fetch time and marked `reused_from`. Every fetch cell reads its snapshot under `data/reference/` when the file exists and calls the network only when it does not. The raw files are not in this repository, so the committed outputs are the record of the run. The LI.FI key is read from `LIFI_API_KEY` in the environment, else from the gitignored `.env`, sent as the `x-lifi-api-key` header, and never printed or written. Sections are numbered.

Snapshots written here (§4, §7): `lifi_api_transfer_status_extension_census_raw.jsonl`, `lifi_api_transfer_status_extension_strata_<chain>_raw.jsonl` (one per source chain, so that every raw file stays under 50 MB) and the flattened `lifi_api_transfer_status_extension.csv` (the 71 columns of `lifi_api_transfer_status_sample_2600.csv` plus `in_census`, `in_strata`, `stratum_integrator`, `stratum_rows_in_data`, `stratum_planned`, `reused_from`). Headline USD is `amount_usd` of the `transfers_priced` view (the provided feed, else CoinGecko); `amount_usd_provided` is the provided feed alone, as in the earlier status CSVs. This notebook is descriptive: the fee-schedule estimator is in query 158.

## 1. Setup: paths, key, HTTP helper with rate-limit pacing, snapshot helpers, flattening (from notebook 03 §1)

In [1]:
import json
import math
import os
import time
import datetime as dt
from collections import Counter
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import requests

from lifi import paths, run, run_sql, build

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 90)

ROOT = paths.REPO_ROOT                 # the repository root, where the gitignored .env lives
DATA_LABEL = "LIFI_DATA_DIR" if os.environ.get("LIFI_DATA_DIR") else "data under the repository root"   # printed instead of a path
REF = paths.reference_dir()            # data/reference in the repository, whatever LIFI_DATA_DIR says
MANIFEST = REF / "snapshot_manifest.json"
LIFI = "https://li.quest/v1"
STATUS_URL = f"{LIFI}/status?txHash=<tx_hash>"
SEED = 20261029                        # every random choice in this notebook
PACE_SECONDS = 0.1                     # sleep after each call; the ratelimit-remaining header does the real pacing (REMAINING_FLOOR)
REMAINING_FLOOR = 3                    # when ratelimit-remaining drops below this, sleep until ratelimit-reset
CENSUS_USD = 10_000
CHAINS = ["ethereum", "arbitrum", "base", "optimism", "polygon", "bnb"]
PRIOR_2600, PRIOR_EDGE = "lifi_api_transfer_status_sample_2600_raw.jsonl", "lifi_api_transfer_status_edge_cases_raw.jsonl"


def read_env_key(name):
    "The key from the environment, else from a KEY=VALUE line in the gitignored .env at the repository root; None if neither has it. Never printed, never written."
    value = os.environ.get(name)
    if value:
        return value
    env_file = ROOT / ".env"
    if env_file.exists():
        for line in env_file.read_text().splitlines():
            line = line.strip()
            if line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            if k.strip() == name:
                return v.strip().strip('"').strip("'") or None
    return None


LIFI_KEY = read_env_key("LIFI_API_KEY")
SESSION = requests.Session()
SESSION.headers["User-Agent"] = "lifi-takehome-analysis/0.1 (analysis notebook 07)"
if LIFI_KEY:
    SESSION.headers["x-lifi-api-key"] = LIFI_KEY
print("data dir:", DATA_LABEL, "| reference snapshots:", REF.relative_to(ROOT), "in the repository | LI.FI key present:", bool(LIFI_KEY))

RATE = {"calls": 0, "n429": 0, "network_errors": 0, "window_waits": 0, "window_wait_s": 0.0, "min_remaining": None, "last_remaining": None}


def now_utc():
    return dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat()


def get_json(url, params=None, pause=PACE_SECONDS, max_tries=8, timeout=60):
    """GET with the session headers. After each response sleeps `pause` seconds, or, when the ratelimit-remaining header is below REMAINING_FLOOR,
    until ratelimit-reset (seconds) has passed; on HTTP 429 sleeps ratelimit-reset plus one and retries; on 5xx and network errors backs off exponentially.
    Returns (status_code, parsed JSON or text, ratelimit_remaining, {"retries_429": n, "window_waits": n}) so the JSONL can record the 429s a call met.
    Status 0 when every try failed on the network."""
    wait, r = 5.0, None
    meta = {"retries_429": 0, "window_waits": 0}
    for attempt in range(max_tries):
        try:
            r = SESSION.get(url, params=params, timeout=timeout)
        except requests.RequestException as e:
            RATE["network_errors"] += 1
            print(f"  network error: {type(e).__name__}; sleeping {wait:.0f}s, attempt {attempt + 1}/{max_tries}")
            time.sleep(wait)
            wait = min(wait * 2, 120)
            if attempt == max_tries - 1:
                return 0, f"network error after {max_tries} tries: {type(e).__name__}", None, meta
            continue
        RATE["calls"] += 1
        remaining, reset = r.headers.get("ratelimit-remaining"), r.headers.get("ratelimit-reset")
        rem_ok = remaining is not None and remaining.isdigit()
        reset_ok = reset is not None and reset.replace(".", "", 1).isdigit()
        if rem_ok:
            RATE["last_remaining"] = int(remaining)
            RATE["min_remaining"] = int(remaining) if RATE["min_remaining"] is None else min(RATE["min_remaining"], int(remaining))
        if r.status_code == 429:
            RATE["n429"] += 1
            meta["retries_429"] += 1
            sleep_s = float(reset) if reset_ok else wait
            print(f"  HTTP 429: sleeping {sleep_s:.0f}s (ratelimit-reset), attempt {attempt + 1}/{max_tries}")
            time.sleep(sleep_s + 1)
            continue
        if r.status_code >= 500:
            print(f"  HTTP {r.status_code}: sleeping {wait:.0f}s, attempt {attempt + 1}/{max_tries}")
            time.sleep(wait)
            wait = min(wait * 2, 120)
            continue
        if rem_ok and int(remaining) < REMAINING_FLOOR and reset_ok:
            RATE["window_waits"] += 1
            RATE["window_wait_s"] += float(reset) + 0.5
            meta["window_waits"] += 1
            time.sleep(float(reset) + 0.5)   # the window is nearly spent; wait for it to reset rather than take a 429
        elif pause:
            time.sleep(pause)
        try:
            return r.status_code, r.json(), remaining, meta
        except ValueError:
            return r.status_code, r.text, remaining, meta
    return r.status_code, r.text, None, meta


def manifest_read():
    return json.loads(MANIFEST.read_text()) if MANIFEST.exists() else {}


def manifest_write(name, source_url, rows, section, fetched_at, note=None):
    m = manifest_read()
    m[name] = {"source_url": source_url, "fetched_at_utc": fetched_at, "rows": int(rows), "notebook_section": section}
    if note:
        m[name]["note"] = note
    MANIFEST.write_text(json.dumps(m, indent=1, sort_keys=True) + "\n")


def snapshot_csv(name, fetch, section, source_url, note=None, read_kwargs=None, fetched_at=None):
    "Read data/reference/<name> if it exists; otherwise call fetch() -> DataFrame, write it and record the manifest entry. `note` and `fetched_at` may be callables evaluated after the fetch."
    path = REF / name
    if path.exists():
        df = pd.read_csv(path, **(read_kwargs or {}))
        print(f"{name}: read snapshot, {len(df)} rows, fetched {manifest_read().get(name, {}).get('fetched_at_utc', '?')} (no network call)")
        return df
    if not LIFI_KEY:
        raise RuntimeError(f"{name} is missing and LIFI_API_KEY is not set: put the key in the environment or in .env, or restore the snapshot")
    print(f"{name}: no snapshot, fetching from {source_url}")
    t0 = now_utc()
    df = fetch()
    df.to_csv(path, index=False)
    t_fetch = fetched_at() if callable(fetched_at) else (fetched_at or t0)
    manifest_write(name, source_url, len(df), section, t_fetch, note() if callable(note) else note)
    print(f"{name}: {len(df)} rows, fetch time recorded {t_fetch} (this cell ran at {t0}), written to {REF.relative_to(ROOT)}/{name} in the repository")
    return pd.read_csv(path, **(read_kwargs or {}))


def jsonl_read(path):
    return [json.loads(line) for line in path.read_text().splitlines()] if path.exists() else []


def fetch_log(recs):
    "What the raw JSONL records show about a fetch: calls, final HTTP statuses, first and last call, observed calls per minute, the minimum ratelimit-remaining, the largest gap between consecutive calls, the HTTP 429s retried before a final response (retries_429, recorded per call here, unlike notebook 03) and the waits for the window to reset."
    ts = sorted(pd.Timestamp(r["fetched_at_utc"]) for r in recs)
    gaps = [(b - a).total_seconds() for a, b in zip(ts, ts[1:])]
    rem = [int(r["ratelimit_remaining"]) for r in recs if str(r.get("ratelimit_remaining") or "").isdigit()]
    minutes = max((ts[-1] - ts[0]).total_seconds() / 60, 1e-9) if ts else 1e-9
    return {"calls": len(recs), "http_statuses": dict(sorted(Counter(r["http_status"] for r in recs).items())), "first_call_utc": ts[0].isoformat() if ts else None, "last_call_utc": ts[-1].isoformat() if ts else None,
            "span_minutes": round(minutes, 1), "calls_per_minute": round(len(recs) / minutes, 1), "min_ratelimit_remaining": min(rem) if rem else None, "max_gap_between_calls_s": max(gaps) if gaps else None,
            "http_429_retried": sum(int(r.get("retries_429") or 0) for r in recs), "window_waits": sum(int(r.get("window_waits") or 0) for r in recs)}


def fetch_span(recs):
    "The fetch log of a list of JSONL records as a sentence for the manifest note."
    f = fetch_log(recs)
    return (f"{f['calls']} calls from {f['first_call_utc']} to {f['last_call_utc']} ({f['calls_per_minute']:.0f} per minute observed, {PACE_SECONDS} s sleep after each call and a wait until ratelimit-reset whenever ratelimit-remaining fell below {REMAINING_FLOOR}: {f['window_waits']} such waits); "
            f"final HTTP statuses {f['http_statuses']}; HTTP 429 responses retried before a final response {f['http_429_retried']} (recorded per call as retries_429); minimum ratelimit-remaining {f['min_ratelimit_remaining']} of 100; largest gap between consecutive calls {f['max_gap_between_calls_s']:.0f} s")


def fee_usd_per_unit(line):
    "USD per smallest token unit of a fee line, from its amount and amountUSD; None when either is missing or zero."
    try:
        amt, usd = int(line.get("amount") or 0), float(line.get("amountUSD") or 0)
    except (TypeError, ValueError):
        return None
    return usd / amt if amt > 0 else None


def flatten_status(ev, rec):
    "One flat row for an event row `ev` (dict with the snake_case columns of transfers_priced) and a raw JSONL record `rec` (its `response` is the parsed body). Verbatim from notebook 03 §1: the same 68 response-and-event columns."
    body = rec.get("response")
    out = {"chain": ev["chain"], "tx_hash": ev["tx_hash"], "log_index": int(ev["log_index"]), "block_time": pd.Timestamp(ev["block_time"]).isoformat(),
           "event_transactionId": ev["transaction_id"], "event_bridge": ev["bridge"], "event_integrator": ev["integrator"], "event_sendingAssetId": ev["sending_asset_id"],
           "event_minAmount": str(ev["min_amount"]), "event_destinationChainId": int(ev["destination_chain_id"]), "event_receiver": ev["receiver"], "event_tx_from": ev["tx_from"],
           "event_hasSourceSwaps": bool(ev["has_source_swaps"]), "event_hasDestinationCall": bool(ev["has_destination_call"]), "amount_usd_provided": ev.get("amount_usd_provided"),
           "http_status": rec.get("http_status"), "fetched_at_utc": rec.get("fetched_at_utc")}
    if not isinstance(body, dict):
        out["api_message"] = str(body)[:200]
        return out
    md_ = body.get("metadata") or {}
    out.update({"api_transactionId": body.get("transactionId"), "api_message": body.get("message"), "status": body.get("status"), "substatus": body.get("substatus"),
                "substatusMessage": body.get("substatusMessage"), "tool": body.get("tool"), "fromAddress": body.get("fromAddress"), "toAddress": body.get("toAddress"),
                "metadata_integrator": md_.get("integrator"),
                "transaction_id_matches": (str(body.get("transactionId") or "").lower() == str(ev["transaction_id"]).lower()) if body.get("transactionId") else None})
    for side in ("sending", "receiving"):
        s = body.get(side) or {}
        tok = s.get("token") or {}
        out.update({f"{side}_txHash": s.get("txHash"), f"{side}_chainId": s.get("chainId"), f"{side}_token_address": tok.get("address"),
                    f"{side}_token_symbol": tok.get("symbol"), f"{side}_token_decimals": tok.get("decimals"), f"{side}_amount": s.get("amount"),
                    f"{side}_amountUSD": s.get("amountUSD"), f"{side}_gasAmountUSD": s.get("gasAmountUSD"), f"{side}_timestamp": s.get("timestamp"),
                    f"{side}_included_tools": "|".join(str(st.get("tool")) for st in (s.get("includedSteps") or []))})
    out["sending_hash_matches"] = (str(out.get("sending_txHash") or "").lower() == ev["tx_hash"].lower()) if out.get("sending_txHash") else None
    batch = body.get("batchStatus")
    if isinstance(batch, list):
        calls = [c for e in batch for c in (e.get("calls") or []) if isinstance(c, dict)]
        out.update({"batch_n": len(batch), "batch_success": "|".join(str(e.get("success")) for e in batch), "batch_calls_n": len(calls),
                    "batch_call_transactionIds": "|".join(str(c.get("transactionId")) for c in calls),
                    "batch_call_statuses": "|".join(f"{c.get('status')}/{c.get('substatus')}" for c in calls),
                    "batch_call_tools": "|".join(str(c.get("tool")) for c in calls)})
    else:
        out.update({"batch_n": None, "batch_success": None, "batch_calls_n": None, "batch_call_transactionIds": None, "batch_call_statuses": None, "batch_call_tools": None})
    q = body.get("quote") or {}
    est = q.get("estimate") or {}
    fees, gas = est.get("feeCosts") or [], est.get("gasCosts") or []
    usd = lambda items: sum(float(i.get("amountUSD") or 0) for i in items)
    lifi_lines = [f for f in fees if str(f.get("name", "")).startswith("LIFI")]
    other_lines = [f for f in fees if not str(f.get("name", "")).startswith("LIFI")]
    split_lifi = split_integ = 0.0
    for f in lifi_lines:
        per_unit, fs = fee_usd_per_unit(f), f.get("feeSplit") or {}
        if per_unit is not None:
            split_lifi += int(fs.get("lifiFee") or 0) * per_unit
            split_integ += int(fs.get("integratorFee") or 0) * per_unit
    out.update({"quote_tool": q.get("tool"), "quote_integrator": q.get("integrator"), "quote_fromAmountUSD": est.get("fromAmountUSD"), "quote_toAmountUSD": est.get("toAmountUSD"),
                "quote_fee_usd": usd(fees) if fees else None,
                "quote_fee_lifi_usd": usd(lifi_lines) if fees else None,
                "quote_fee_other_usd": usd(other_lines) if fees else None,
                "quote_fee_names": "|".join(str(f.get("name")) for f in fees),
                "quote_fee_lifi_percentages": "|".join(str(f.get("percentage")) for f in lifi_lines),
                "quote_fee_lifi_included": "|".join(str(f.get("included")) for f in lifi_lines),
                "fee_split_lifi_usd": split_lifi if lifi_lines else None,
                "fee_split_integrator_usd": split_integ if lifi_lines else None,
                "fee_split_recipients": "|".join(str(r.get("name")) for f in lifi_lines for r in ((f.get("feeSplit") or {}).get("recipients") or [])),
                "quote_gas_usd": usd(gas) if gas else None})
    return out


STR_COLS = ["tx_hash", "event_transactionId", "event_sendingAssetId", "event_minAmount", "event_receiver", "event_tx_from", "api_transactionId", "sending_txHash", "receiving_txHash",
            "sending_token_address", "receiving_token_address", "sending_amount", "receiving_amount", "fromAddress", "toAddress", "status", "substatus", "substatusMessage",
            "tool", "quote_tool", "quote_fee_names", "quote_fee_lifi_percentages", "quote_fee_lifi_included", "fee_split_recipients", "sending_included_tools", "receiving_included_tools",
            "batch_success", "batch_call_transactionIds", "batch_call_statuses", "batch_call_tools", "lookup", "lookup_value", "sample_design", "api_message", "metadata_integrator", "quote_integrator",
            "stratum_integrator", "reused_from", "event_integrator"]
NUM_COLS = ["sending_amountUSD", "receiving_amountUSD", "sending_gasAmountUSD", "receiving_gasAmountUSD", "quote_fromAmountUSD", "quote_toAmountUSD", "quote_fee_usd",
            "quote_fee_lifi_usd", "quote_fee_other_usd", "fee_split_lifi_usd", "fee_split_integrator_usd", "quote_gas_usd", "amount_usd_provided"]


def read_status_csv(name):
    df = pd.read_csv(REF / name, dtype={c: str for c in STR_COLS}, keep_default_na=True)
    for c in NUM_COLS:
        if c in df:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    for c in ("sending_chainId", "receiving_chainId", "batch_n", "batch_calls_n", "log_index", "event_destinationChainId", "http_status", "stratum_rows_in_data", "stratum_planned"):
        if c in df:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
    return df


data dir: LIFI_DATA_DIR | reference snapshots: data/reference in the repository | LI.FI key present: True


## 2. The swap rows, the headline USD and the committed status snapshots

The population is every event with `hasSourceSwaps = true`, read from the `transfers_priced` view of `analysis/lifi.duckdb` (`uv run lifi-build-db`), whose `amount_usd` is the headline USD column (the provided feed, else the CoinGecko snapshot) and `amount_usd_provided` the provided feed alone. The raw JSONLs of notebook 03 are indexed by transaction hash: a hash already there is reused, not refetched. The status endpoint keys on the hash and returns the first event of a transaction (03 §5.2), so the fetch unit is the hash and the two events of a two-event transaction share one record.

In [2]:
sw = run_sql("""
    SELECT chain, block_time, block_number, tx_hash, log_index, tx_from, transaction_id, bridge, integrator, referrer, sending_asset_id, receiver, min_amount,
           destination_chain_id, has_source_swaps, has_destination_call, amount_usd, amount_usd_provided, price_source, symbol
    FROM transfers_priced
    WHERE has_source_swaps
    ORDER BY chain, block_time, log_index
""")
n_all = int(run_sql("SELECT count(*) AS n FROM transfers")["n"].iloc[0])
sw["integrator"] = sw["integrator"].fillna("")          # the two empty-integrator rows: the empty string is an integrator string here (03 §5.6)
sw["day"] = pd.to_datetime(sw["block_time"]).dt.date
sw["key"] = list(zip(sw["chain"], sw["tx_hash"], sw["log_index"].astype(int)))
assert sw["key"].is_unique and sw["has_source_swaps"].all()
print(f"swap rows: {len(sw):,} of {n_all:,} events; headline USD {sw['amount_usd'].sum():,.2f}; unpriced under the headline column {int(sw['amount_usd'].isna().sum())}; "
      f"priced by CoinGecko only {int((sw['price_source'] == 'coingecko').sum())}; distinct hashes {sw['tx_hash'].nunique():,} (two-event transactions {int((sw.groupby('tx_hash').size() > 1).sum())}); "
      f"integrator strings {sw['integrator'].nunique()} (empty string on {int((sw['integrator'] == '').sum())} rows)")
print("swap rows by chain:", sw["chain"].value_counts().to_dict())

# The committed raw records, by hash: the 2,600-event sample (600 + uniform 2,000) first, then the edge cases' plain hash lookups.
prior = {}
for name, recs in ((PRIOR_2600, jsonl_read(REF / PRIOR_2600)), (PRIOR_EDGE, [r for r in jsonl_read(REF / PRIOR_EDGE) if r.get("lookup") == "tx_hash"])):
    n_new = 0
    for r in recs:
        h = r.get("tx_hash") or r.get("lookup_value")
        if h not in prior:
            prior[h] = (name, r)
            n_new += 1
    print(f"{name}: {len(recs)} hash records, {n_new} hashes not already indexed")
print(f"hashes reusable from the committed snapshots: {len(prior):,}; of them on swap rows: {int(sw['tx_hash'].isin(prior).sum()):,} rows")


swap rows: 84,679 of 124,858 events; headline USD 64,534,185.70; unpriced under the headline column 4; priced by CoinGecko only 354; distinct hashes 84,668 (two-event transactions 11); integrator strings 584 (empty string on 2 rows)
swap rows by chain: {'base': 25623, 'ethereum': 19865, 'bnb': 18696, 'arbitrum': 11008, 'polygon': 5408, 'optimism': 4079}


lifi_api_transfer_status_sample_2600_raw.jsonl: 2600 hash records, 2600 hashes not already indexed
lifi_api_transfer_status_edge_cases_raw.jsonl: 283 hash records, 282 hashes not already indexed
hashes reusable from the committed snapshots: 2,882; of them on swap rows: 1,999 rows


## 3. Selection: the census and the integrator strata

Two designs over the swap rows, seed 20261029, each a fixed function of the data:

- `census_swap_10k`: every swap row with `amount_usd >= 10,000`. No draw.
- `integrator_strata`: per raw integrator string (underscore-prefixed, test and empty strings included; no normalisation), over its swap rows: 30 events when it has 100 or more, 10 when 10 to 99, every row when under 10. The draw is **systematic**: the integrator's rows are sorted by (`destination_chain_id`, `sending_asset_id`, `block_time`, `log_index`); with `N` rows and `n` planned, the step is `k = N / n`, one start `u ~ Uniform[0, k)` is drawn from `numpy.random.default_rng(20261029)` (integrators in sorted order, one draw each), and the rows at positions `floor(u + i·k)` for `i = 0 … n−1` are taken. The sort puts the rows of one destination chain together, within it one token together, within that in time order, so the `n` picks fall across destination chains, tokens and days in proportion to the rows each holds (a proportional implicit stratification): a rate that depends on route, token or day shows up as a mixture of rates in the stratum instead of hiding behind one route's draw.

A row drawn for a stratum that is also in the census is one event, fetched once, with `in_census` and `in_strata` both true; it counts toward its integrator's stratum. The planned-call table below says what is fetched, what is reused from the committed snapshots and what is in both designs.

In [3]:
census_idx = sw.index[sw["amount_usd"] >= CENSUS_USD]
counts = sw.groupby("integrator").size()


def planned_for(n_rows):
    return 30 if n_rows >= 100 else (10 if n_rows >= 10 else int(n_rows))


def tier_for(n_rows):
    return "30 of 100+" if n_rows >= 100 else ("10 of 10-99" if n_rows >= 10 else "all under 10")


sw["tier"] = sw["integrator"].map(counts).map(tier_for)
rng = np.random.default_rng(SEED)
strata_rows, starts = [], {}
for integ in sorted(counts.index):
    g = sw[sw["integrator"] == integ].sort_values(["destination_chain_id", "sending_asset_id", "block_time", "log_index"], kind="stable")
    N, n = len(g), planned_for(len(g))
    if n >= N:
        picked = g.index
    else:
        k = N / n
        u = rng.uniform(0, k)
        starts[integ] = round(float(u), 4)
        positions = np.floor(u + k * np.arange(n)).astype(int)
        assert positions.max() < N and len(set(positions)) == n
        picked = g.index[positions]
    strata_rows.append(picked)
strata_idx = pd.Index(np.concatenate([np.asarray(i) for i in strata_rows]))
assert strata_idx.is_unique

sel = sw.loc[sorted(set(census_idx) | set(strata_idx))].copy()
sel["in_census"] = sel.index.isin(census_idx)
sel["in_strata"] = sel.index.isin(strata_idx)
sel["stratum_integrator"] = sel["integrator"]
sel["stratum_rows_in_data"] = sel["integrator"].map(counts).astype(int)
sel["stratum_planned"] = sel["stratum_rows_in_data"].map(planned_for).astype(int)
sel["tier"] = sel["stratum_rows_in_data"].map(tier_for)
sel["sample_design"] = np.where(sel["in_census"], "census_swap_10k", "integrator_strata")     # the raw file the record is fetched into; membership is in_census / in_strata
sel["reused_from"] = sel["tx_hash"].map(lambda h: prior[h][0] if h in prior else None)
sel = sel.sort_values(["chain", "block_time", "log_index"], kind="stable")
hashes_to_fetch = sel.loc[sel["reused_from"].isna(), "tx_hash"].drop_duplicates()
n_tiers = counts.map(tier_for).value_counts()
print(f"census: {len(census_idx):,} rows (swap rows at headline USD >= {CENSUS_USD:,}), {sw.loc[census_idx, 'amount_usd'].sum():,.2f} USD = {sw.loc[census_idx, 'amount_usd'].sum() / sw['amount_usd'].sum():.1%} of swap-row headline USD")
print(f"strata: {len(counts)} integrator strings; tiers {n_tiers.to_dict()}; planned events {int(counts.map(planned_for).sum()):,} (30-tier {30 * n_tiers.get('30 of 100+', 0)}, 10-tier {10 * n_tiers.get('10 of 10-99', 0)}, all-rows tier {int(counts[counts < 10].sum())}); drawn {len(strata_idx):,}")
print(f"selected events: {len(sel):,} = census {int(sel['in_census'].sum()):,} + strata {int(sel['in_strata'].sum()):,} − in both {int((sel['in_census'] & sel['in_strata']).sum())}; distinct hashes {sel['tx_hash'].nunique():,}; "
      f"reused from committed snapshots {int(sel['reused_from'].notna().sum())} rows ({sel['reused_from'].value_counts().to_dict()}); hashes to fetch {len(hashes_to_fetch):,}")


def plan_table(frame, by):
    t = frame.groupby(by, observed=True).agg(events=("tx_hash", "size"), distinct_hashes=("tx_hash", "nunique"), in_census=("in_census", "sum"), in_strata=("in_strata", "sum"),
                                             in_both=("in_census", lambda s: int((s & frame.loc[s.index, "in_strata"]).sum())),
                                             reused_2600=("reused_from", lambda s: int((s == PRIOR_2600).sum())), reused_edge=("reused_from", lambda s: int((s == PRIOR_EDGE).sum())),
                                             to_fetch=("reused_from", lambda s: int(frame.loc[s.index][s.isna()]["tx_hash"].nunique())), headline_usd=("amount_usd", "sum"))
    return t.round({"headline_usd": 2})


print("planned calls by design (a row in both designs is one event, fetched once):")
by_design = pd.concat({"census_swap_10k": plan_table(sel[sel["in_census"]].assign(_="x"), "_"), "integrator_strata": plan_table(sel[sel["in_strata"]].assign(_="x"), "_"),
                       "union (every selected event)": plan_table(sel.assign(_="x"), "_")}).droplevel(1)
display(by_design)
print("by tier (strata rows only; rows_in_data is the tier's swap rows in the data, integrators the strings in the tier):")
by_tier = plan_table(sel[sel["in_strata"]], "tier")
by_tier.insert(0, "integrators", sel[sel["in_strata"]].groupby("tier")["integrator"].nunique())
by_tier.insert(1, "rows_in_data", sw.groupby("tier").size())
by_tier.insert(2, "planned", sel[sel["in_strata"]].groupby("tier")["stratum_planned"].sum())
display(by_tier.loc[["30 of 100+", "10 of 10-99", "all under 10"]])
print("by source chain (every selected event; rows_in_data is the chain's swap rows):")
by_chain = plan_table(sel, "chain")
by_chain.insert(0, "rows_in_data", sw["chain"].value_counts())
display(by_chain.sort_values("events", ascending=False))


census: 751 rows (swap rows at headline USD >= 10,000), 41,895,314.75 USD = 64.9% of swap-row headline USD
strata: 584 integrator strings; tiers {'all under 10': 391, '10 of 10-99': 138, '30 of 100+': 55}; planned events 4,165 (30-tier 1650, 10-tier 1380, all-rows tier 1135); drawn 4,165
selected events: 4,868 = census 751 + strata 4,165 − in both 48; distinct hashes 4,866; reused from committed snapshots 133 rows ({'lifi_api_transfer_status_sample_2600_raw.jsonl': 98, 'lifi_api_transfer_status_edge_cases_raw.jsonl': 35}); hashes to fetch 4,735
planned calls by design (a row in both designs is one event, fetched once):


,events,distinct_hashes,in_census,in_strata,in_both,reused_2600,reused_edge,to_fetch,headline_usd
census_swap_10k,751,751,751,48,48,19,13,719,41895314.75
integrator_strata,4165,4163,48,4165,48,79,24,4062,6539813.92
union (every selected event),4868,4866,751,4165,48,98,35,4735,42914020.91


by tier (strata rows only; rows_in_data is the tier's swap rows in the data, integrators the strings in the tier):


,integrators,rows_in_data,planned,events,distinct_hashes,in_census,in_strata,in_both,reused_2600,reused_edge,to_fetch,headline_usd
tier,,,,,,,,,,,,
30 of 100+,55,79300,49500,1650,1650,14,1650,14,33,3,1614,947096.57
10 of 10-99,138,4244,13800,1380,1379,26,1380,26,26,4,1350,1998150.64
all under 10,391,1135,5329,1135,1134,8,1135,8,20,17,1098,3594566.72


by source chain (every selected event; rows_in_data is the chain's swap rows):


,rows_in_data,events,distinct_hashes,in_census,in_strata,in_both,reused_2600,reused_edge,to_fetch,headline_usd
chain,,,,,,,,,,
ethereum,19865,1440,1438,397,1071,28,24,18,1398,22882460.74
base,25623,1150,1150,74,1080,4,20,1,1129,2240454.75
bnb,18696,976,976,89,891,4,19,2,955,3584593.16
arbitrum,11008,774,774,159,626,11,17,9,748,13555574.69
polygon,5408,388,388,19,369,0,11,3,374,332991.76
optimism,4079,140,140,13,128,1,7,2,131,317945.82


In [4]:
# Per integrator: the plan for every stratum (rows in data, planned, census rows, reused, to fetch) and the spread of the systematic draw over destination chain, token and day.
per_integ = sel.groupby("integrator").agg(tier=("tier", "first"), rows_in_data=("stratum_rows_in_data", "first"), planned=("stratum_planned", "first"), in_strata=("in_strata", "sum"),
                                          in_census=("in_census", "sum"), events=("tx_hash", "size"), reused=("reused_from", lambda s: int(s.notna().sum())),
                                          to_fetch=("reused_from", lambda s: int(s.isna().sum())))
spread = sel[sel["in_strata"]].groupby("integrator").agg(dest_chains_drawn=("destination_chain_id", "nunique"), tokens_drawn=("sending_asset_id", "nunique"), days_drawn=("day", "nunique"))
avail = sw.groupby("integrator").agg(dest_chains_in_data=("destination_chain_id", "nunique"), tokens_in_data=("sending_asset_id", "nunique"), days_in_data=("day", "nunique"))
per_integ = per_integ.join(spread).join(avail)
per_integ["systematic_start"] = per_integ.index.map(starts)
per_integ.index = per_integ.index.map(lambda s: "(empty)" if s == "" else s)
assert (per_integ["in_strata"] == per_integ["planned"]).all(), "every stratum holds exactly its planned events"
print(f"{len(per_integ)} strata; every stratum holds its planned count: {(per_integ['in_strata'] == per_integ['planned']).all()}; strata with a census row: {int((per_integ['in_census'] > 0).sum())}")
print("the 30-event tier, by swap rows in the data (dest_chains/tokens/days: distinct values among the 30 drawn against the distinct values the integrator's swap rows hold):")
display(per_integ[per_integ["tier"] == "30 of 100+"].sort_values("rows_in_data", ascending=False))
print("the 10-event tier (first 20 by rows in data) and the all-rows tier, summarised:")
display(per_integ[per_integ["tier"] == "10 of 10-99"].sort_values("rows_in_data", ascending=False).head(20))
display(per_integ.groupby("tier")[["rows_in_data", "planned", "in_strata", "in_census", "events", "reused", "to_fetch"]].sum().loc[["30 of 100+", "10 of 10-99", "all under 10"]])
print("the 30-event tier's draws against the data, pooled: destination chains covered", int(sel[(sel['in_strata']) & (sel['tier'] == '30 of 100+')]['destination_chain_id'].nunique()), "of", int(sw[sw['tier'] == '30 of 100+']['destination_chain_id'].nunique()),
      "; sending tokens", int(sel[(sel['in_strata']) & (sel['tier'] == '30 of 100+')]['sending_asset_id'].nunique()), "of", int(sw[sw['tier'] == '30 of 100+']['sending_asset_id'].nunique()))


584 strata; every stratum holds its planned count: True; strata with a census row: 53
the 30-event tier, by swap rows in the data (dest_chains/tokens/days: distinct values among the 30 drawn against the distinct values the integrator's swap rows hold):


,tier,rows_in_data,planned,in_strata,in_census,events,reused,to_fetch,dest_chains_drawn,tokens_drawn,days_drawn,dest_chains_in_data,tokens_in_data,days_in_data,systematic_start
integrator,,,,,,,,,,,,,,,
jumper.exchange,30 of 100+,20877,30,30,296,325,14,311,13,10,7,63,133,7,221.2583
rabbykmsv2,30 of 100+,7758,30,30,64,94,1,93,8,11,7,29,46,7,145.1845
metamask-bridge,30 of 100+,6853,30,30,34,64,2,62,8,15,7,16,35,7,193.5726
base-app,30 of 100+,5765,30,30,1,31,1,30,9,8,7,11,33,7,42.1018
_binancewallet,30 of 100+,5722,30,30,13,43,0,43,7,12,7,17,36,7,61.8906
phantom,30 of 100+,5230,30,30,39,69,2,67,7,10,7,9,36,7,163.3389
lifi-api,30 of 100+,3464,30,30,9,39,0,39,10,10,7,44,41,7,87.6149
tangem,30 of 100+,2621,30,30,27,56,1,55,12,13,7,25,69,7,51.4197
bitget,30 of 100+,2534,30,30,0,30,1,29,5,8,7,8,18,7,7.3930


the 10-event tier (first 20 by rows in data) and the all-rows tier, summarised:


,tier,rows_in_data,planned,in_strata,in_census,events,reused,to_fetch,dest_chains_drawn,tokens_drawn,days_drawn,dest_chains_in_data,tokens_in_data,days_in_data,systematic_start
integrator,,,,,,,,,,,,,,,
gas-sweeper,10 of 10-99,98,10,10,0,10,0,10,3,8,5,3,18,7,6.6692
gmgntestenv,10 of 10-99,97,10,10,0,10,0,10,3,4,7,4,6,7,7.6980
tangem-pay-prod,10 of 10-99,91,10,10,0,10,1,9,1,4,6,1,5,7,5.8426
jumperadvanced,10 of 10-99,85,10,10,2,12,0,12,8,7,6,16,17,7,4.1799
exa_app,10 of 10-99,77,10,10,0,10,1,9,4,7,5,5,13,7,0.9825
ambire-extension-prod,10 of 10-99,74,10,10,0,10,0,10,7,5,5,11,13,7,4.2809
debot,10 of 10-99,72,10,10,0,10,0,10,5,5,2,6,6,5,0.4674
mbridge,10 of 10-99,68,10,10,1,11,0,11,5,4,5,10,20,7,2.5186
Curvy-Staging,10 of 10-99,68,10,10,0,10,1,9,6,5,5,11,8,6,5.2580


,rows_in_data,planned,in_strata,in_census,events,reused,to_fetch
tier,,,,,,,
30 of 100+,79300,1650,1650,621,2257,59,2198
10 of 10-99,4244,1380,1380,122,1476,37,1439
all under 10,1135,1135,1135,8,1135,37,1098


the 30-event tier's draws against the data, pooled: destination chains covered 31 of 66 ; sending tokens 43 of 201


## 4. Fetch: one `/v1/status?txHash=<hash>` call per hash not already in a committed snapshot

Calls go to `lifi_api_transfer_status_extension_census_raw.jsonl` for hashes in the census (a hash in both designs is fetched once, into this file) and to `lifi_api_transfer_status_extension_strata_<chain>_raw.jsonl`, one file per source chain, for the strata-only hashes, so that every raw file stays under 50 MB (the 2,600-event file is 33 MB). Each record carries the design, the event keys, `request_params` (no key), the fetch time, the HTTP status, `ratelimit_remaining`, `retries_429` (HTTP 429s met before the final response) and `window_waits`, then the raw response. The loop appends as responses arrive and skips hashes already in any of the files, so a run that stops resumes where it stopped. The flattened CSV is written once every hash has a record: one row per selected event, reused rows taking the committed record with its original fetch time and `reused_from` naming the file.

In [5]:
EXT_CSV = "lifi_api_transfer_status_extension.csv"
CENSUS_JSONL = REF / "lifi_api_transfer_status_extension_census_raw.jsonl"
STRATA_JSONL = {c: REF / f"lifi_api_transfer_status_extension_strata_{c}_raw.jsonl" for c in CHAINS}
NEW_JSONLS = [CENSUS_JSONL] + [STRATA_JSONL[c] for c in CHAINS]
EXTRA_COLS = ["sample_design", "lookup", "lookup_value", "in_census", "in_strata", "stratum_integrator", "stratum_rows_in_data", "stratum_planned", "reused_from"]


def planned_calls():
    "One call per hash to fetch, in selection order: (file, chain, hash, the first event's log_index and transactionId, design flags)."
    calls, seen = [], set()
    for ev in sel[sel["reused_from"].isna()].itertuples(index=False):
        if ev.tx_hash in seen:
            continue
        seen.add(ev.tx_hash)
        calls.append({"path": CENSUS_JSONL if ev.in_census else STRATA_JSONL[ev.chain], "chain": ev.chain, "tx_hash": ev.tx_hash, "log_index": int(ev.log_index), "event_transactionId": ev.transaction_id,
                      "sample_design": ev.sample_design, "in_census": bool(ev.in_census), "in_strata": bool(ev.in_strata)})
    return calls


def new_records():
    "Every record in the new raw files, by hash."
    return {r["tx_hash"]: r for p in NEW_JSONLS for r in jsonl_read(p)}


def fetch_missing(calls):
    done = new_records()
    todo = [c for c in calls if c["tx_hash"] not in done]
    print(f"{len(calls):,} calls planned, {len(done):,} already in the new raw files, {len(todo):,} to fetch (~{len(todo) / 85:.0f} min at 85 per minute)")
    t_start, handles = time.time(), {}
    try:
        for i, c in enumerate(todo):
            status, body, remaining, meta = get_json(f"{LIFI}/status", params={"txHash": c["tx_hash"]})
            rec = {"lookup": "tx_hash", "chain": c["chain"], "lookup_value": c["tx_hash"], "bridge_param": None, "request_params": {"txHash": c["tx_hash"]},
                   "tx_hash": c["tx_hash"], "log_index": c["log_index"], "event_transactionId": c["event_transactionId"], "sample_design": c["sample_design"], "in_census": c["in_census"], "in_strata": c["in_strata"],
                   "fetched_at_utc": now_utc(), "http_status": status, "ratelimit_remaining": remaining, "retries_429": meta["retries_429"], "window_waits": meta["window_waits"], "response": body}
            fh = handles.get(c["path"])
            if fh is None:
                fh = handles[c["path"]] = c["path"].open("a")
            fh.write(json.dumps(rec) + "\n")
            fh.flush()
            done[c["tx_hash"]] = rec
            if (i + 1) % 250 == 0 or i + 1 == len(todo):
                print(f"  {i + 1}/{len(todo)} fetched, {(time.time() - t_start) / 60:.1f} min, HTTP 429 so far {RATE['n429']}, window waits {RATE['window_waits']} ({RATE['window_wait_s']:.0f} s), min ratelimit-remaining seen {RATE['min_remaining']}")
    finally:
        for fh in handles.values():
            fh.close()
    return done


def fetch_extension():
    done = fetch_missing(planned_calls())
    missing = [h for h in sel["tx_hash"].unique() if h not in done and h not in prior]
    assert not missing, f"{len(missing)} hashes without a record"
    rows = []
    for ev in sel.to_dict("records"):
        name, rec = prior[ev["tx_hash"]] if ev["tx_hash"] in prior else (None, done[ev["tx_hash"]])
        row = flatten_status(ev, rec)
        row.update({"sample_design": ev["sample_design"], "lookup": "tx_hash", "lookup_value": ev["tx_hash"], "in_census": bool(ev["in_census"]), "in_strata": bool(ev["in_strata"]),
                    "stratum_integrator": ev["stratum_integrator"], "stratum_rows_in_data": int(ev["stratum_rows_in_data"]), "stratum_planned": int(ev["stratum_planned"]), "reused_from": name})
        rows.append(row)
    out = pd.DataFrame(rows)
    base_cols = [c for c in pd.read_csv(REF / "lifi_api_transfer_status_sample_2600.csv", nrows=0).columns]   # the 71 columns, in the 2600 file's order
    assert set(base_cols) <= set(out.columns), sorted(set(base_cols) - set(out.columns))
    return out[base_cols + [c for c in EXTRA_COLS if c not in base_cols]]


def ext_note():
    recs = [r for p in NEW_JSONLS for r in jsonl_read(p)]
    return (f"two designs over the {len(sw):,} swap rows (has_source_swaps), seed {SEED}: census_swap_10k = every row with headline amount_usd >= {CENSUS_USD:,} ({int(sel['in_census'].sum())} events); "
            f"integrator_strata = per raw integrator string 30 events when it has 100+ swap rows, 10 when 10-99, every row under 10, a systematic draw over the rows sorted by (destination_chain_id, sending_asset_id, block_time) "
            f"({int(sel['in_strata'].sum())} events over {len(counts)} strings; {int((sel['in_census'] & sel['in_strata']).sum())} events in both designs); {len(sel)} rows, one per event; "
            f"{int(sel['reused_from'].notna().sum())} rows reuse the committed record of their hash ({sel['reused_from'].value_counts().to_dict()}) with its original fetch time; the rest from {len(recs)} new calls: {fetch_span(recs)}; "
            f"raw responses in lifi_api_transfer_status_extension_census_raw.jsonl and lifi_api_transfer_status_extension_strata_<chain>_raw.jsonl")


ext = snapshot_csv(EXT_CSV, fetch_extension, "07 §4", STATUS_URL, note=ext_note, fetched_at=lambda: min(r["fetched_at_utc"] for p in NEW_JSONLS for r in jsonl_read(p)))
ext = read_status_csv(EXT_CSV)
manifest = manifest_read()
for p in NEW_JSONLS:
    if p.exists() and p.name not in manifest:
        recs = jsonl_read(p)
        manifest_write(p.name, STATUS_URL, len(recs), "07 §4", min(r["fetched_at_utc"] for r in recs),
                       note=(f"raw JSON responses, one line per call ({'census hashes, including those also drawn for a stratum' if p == CENSUS_JSONL else 'strata-only hashes on ' + p.name.split('_strata_')[1].replace('_raw.jsonl', '')}), "
                             f"with sample_design, in_census, in_strata, the event keys, request parameters (no key), HTTP status, ratelimit-remaining, retries_429 and window_waits; {fetch_span(recs)}"))
print(f"{EXT_CSV}: {ext.shape}; in_census {int(ext['in_census'].sum())}, in_strata {int(ext['in_strata'].sum())}, both {int((ext['in_census'] & ext['in_strata']).sum())}; reused_from {ext['reused_from'].value_counts(dropna=False).to_dict()}; HTTP statuses {ext['http_status'].value_counts().to_dict()}")
assert len(ext) == len(sel) and set(zip(ext["chain"], ext["tx_hash"], ext["log_index"].astype(int))) == set(sel["key"])
print(f"calls this run: {RATE['calls']}, HTTP 429: {RATE['n429']}, network errors: {RATE['network_errors']}, window waits: {RATE['window_waits']}, min ratelimit-remaining seen: {RATE['min_remaining']}")


lifi_api_transfer_status_extension.csv: read snapshot, 4868 rows, fetched 2026-11-01T12:05:46+00:00 (no network call)
lifi_api_transfer_status_extension.csv: (4868, 77); in_census 751, in_strata 4165, both 48; reused_from {nan: 4735, 'lifi_api_transfer_status_sample_2600_raw.jsonl': 98, 'lifi_api_transfer_status_edge_cases_raw.jsonl': 35}; HTTP statuses {np.int64(200): 4868}
calls this run: 0, HTTP 429: 0, network errors: 0, window waits: 0, min ratelimit-remaining seen: None


## 5. The fetch log, from the raw JSONLs

As 03 §11: calls, final HTTP statuses, first and last call, span, observed calls per minute, the minimum `ratelimit-remaining`, the largest gap between consecutive calls; and, new here, the HTTP 429 responses met before a final response (`retries_429`, recorded per call) and the waits for the window to reset (`window_waits`). The reused records are not calls of this notebook and are not in this table.

In [6]:
recs_by_file = {p.name: jsonl_read(p) for p in NEW_JSONLS if p.exists()}
all_new = [r for recs in recs_by_file.values() for r in recs]
cols = {"census (extension_census_raw)": recs_by_file.get(CENSUS_JSONL.name, [])}
cols.update({f"strata {c}": recs_by_file.get(STRATA_JSONL[c].name, []) for c in CHAINS if STRATA_JSONL[c].name in recs_by_file})
cols["all new calls"] = all_new
assert all_new, "no new raw records: the fetch of §4 did not run"
fetch_log_table = pd.DataFrame({k: fetch_log(v) for k, v in cols.items() if v})
display(fetch_log_table)
rem = pd.Series([int(r["ratelimit_remaining"]) for r in all_new if str(r.get("ratelimit_remaining") or "").isdigit()])
print("ratelimit-remaining over all new calls:", rem.describe(percentiles=[0.01, 0.05, 0.1, 0.5]).round(1).to_dict(), "; calls with ratelimit-remaining below", REMAINING_FLOOR, ":", int((rem < REMAINING_FLOOR).sum()))
ts = pd.Series(sorted(pd.Timestamp(r["fetched_at_utc"]) for r in all_new))
per_min = ts.dt.floor("min").value_counts().sort_index()
print(f"calls per clock minute: max {int(per_min.max())}, median {per_min.median():.0f}, minutes with calls {len(per_min)}; minutes with more than 100 calls: {int((per_min > 100).sum())}")
print(f"runs (gaps above 60 s between consecutive calls split them): {int((ts.diff().dt.total_seconds() > 60).sum()) + 1}; gaps above 60 s: {[round(g) for g in ts.diff().dt.total_seconds().dropna() if g > 60]}")
print("records by design and HTTP status:", pd.crosstab([r["sample_design"] for r in all_new], [r["http_status"] for r in all_new]).to_dict())
print("records whose response is not a dict (an error body):", sum(1 for r in all_new if not isinstance(r.get("response"), dict)))


,census (extension_census_raw),strata ethereum,strata arbitrum,strata base,strata optimism,strata polygon,strata bnb,all new calls
calls,719,1019,598,1058,118,355,868,4735
http_statuses,{200: 719},{200: 1019},{200: 598},{200: 1058},{200: 118},{200: 355},{200: 868},{200: 4735}
first_call_utc,2026-11-01T12:05:58+00:00,2026-11-01T12:39:40+00:00,2026-11-01T12:05:46+00:00,2026-11-01T12:14:53+00:00,2026-11-01T12:54:46+00:00,2026-11-01T12:56:31+00:00,2026-11-01T12:28:45+00:00,2026-11-01T12:05:46+00:00
last_call_utc,2026-11-01T13:00:48+00:00,2026-11-01T12:54:46+00:00,2026-11-01T12:14:52+00:00,2026-11-01T12:28:44+00:00,2026-11-01T12:56:30+00:00,2026-11-01T13:00:54+00:00,2026-11-01T12:39:39+00:00,2026-11-01T13:00:54+00:00
span_minutes,54.8,15.1,9.1,13.8,1.7,4.4,10.9,55.1
calls_per_minute,13.1,67.5,65.7,76.4,68.1,81.0,79.6,85.9
min_ratelimit_remaining,3,2,4,8,8,8,4,2
max_gap_between_calls_s,80.0,6.0,4.0,9.0,5.0,3.0,5.0,9.0
http_429_retried,0,0,0,0,0,0,0,0
window_waits,0,7,0,0,0,0,0,7


ratelimit-remaining over all new calls: {'count': 4735.0, 'mean': 55.7, 'std': 25.9, 'min': 2.0, '1%': 6.0, '5%': 14.0, '10%': 20.0, '50%': 56.0, 'max': 99.0} ; calls with ratelimit-remaining below 3 : 7
calls per clock minute: max 104, median 87, minutes with calls 56; minutes with more than 100 calls: 2
runs (gaps above 60 s between consecutive calls split them): 1; gaps above 60 s: []
records by design and HTTP status: {200: {'census_swap_10k': 719, 'integrator_strata': 4016}}
records whose response is not a dict (an error body): 0


## 6. Results by design (descriptive)

The rows of `lifi_api_transfer_status_extension.csv`, read through the named queries 143 to 148 on the `ref_lifi_api_transfer_status_extension` table of the DuckDB file (`uv run lifi-build-db` loads every CSV under `data/reference/`). If the file was just written in §4 the table does not exist yet and the database is rebuilt here once. Rates follow query `76_status_lifi_fee_rate_distribution`: LI.FI rate = line `percentage` × `lifiFee` / line amount in bps, the integrator rate likewise with `integratorFee`. No weighting and no schedule estimate here (see query 158).

In [7]:
with duckdb.connect(str(paths.db_path()), read_only=True) as con:
    have_table = "ref_lifi_api_transfer_status_extension" in {t[0] for t in con.execute("SHOW TABLES").fetchall()}
if not have_table:
    print("the database has no ref_lifi_api_transfer_status_extension table yet: rebuilding analysis/lifi.duckdb from the CSVs and the snapshots")
    build(out=None)
n_ext = int(run_sql("SELECT count(*) AS n FROM ref_lifi_api_transfer_status_extension")["n"].iloc[0])
assert n_ext == len(ext), (n_ext, len(ext))
print(f"ref_lifi_api_transfer_status_extension: {n_ext:,} rows, equal to the CSV")


ref_lifi_api_transfer_status_extension: 4,868 rows, equal to the CSV


### 6.1 Coverage: per stratum and per (design, chain) (queries `143_status_extension_coverage_by_stratum`, `144_status_extension_coverage_by_design_and_chain`)

In [8]:
cov = run("143_status_extension_coverage_by_stratum")
print(f"strata: {len(cov)}; every stratum holds its planned events: {bool((cov['events_in_snapshot'] == cov['planned']).all())}; events {int(cov['events_in_snapshot'].sum()):,} (reused from the 2,600 {int(cov['reused_from_2600'].sum())}, from the edge cases {int(cov['reused_from_edge_cases'].sum())}, fetched {int(cov['fetched_new'].sum()):,}); "
      f"HTTP 200 on {int(cov['http_200'].sum()):,}; with a LIFI-named line {int(cov['with_lifi_line'].sum()):,}")
print("by tier:")
display(cov.groupby("tier")[["rows_in_data", "planned", "events_in_snapshot", "in_census_too", "reused_from_2600", "reused_from_edge_cases", "fetched_new", "http_200", "completed", "with_lifi_line"]].sum().loc[["30 of 100+", "10 of 10-99", "all under 10"]])
print("the 30-event tier:")
display(cov[cov["tier"] == "30 of 100+"].set_index("integrator"))
print("by design and source chain (query 144):")
display(run("144_status_extension_coverage_by_design_and_chain").set_index(["design", "chain"]))


strata: 584; every stratum holds its planned events: True; events 4,165 (reused from the 2,600 79, from the edge cases 24, fetched 4,062); HTTP 200 on 4,165; with a LIFI-named line 3,812
by tier:


,rows_in_data,planned,events_in_snapshot,in_census_too,reused_from_2600,reused_from_edge_cases,fetched_new,http_200,completed,with_lifi_line
tier,,,,,,,,,,
30 of 100+,79300,1650,1650,14,33,3,1614,1650,1626,1487
10 of 10-99,4244,1380,1380,26,26,4,1350,1380,1349,1242
all under 10,1135,1135,1135,8,20,17,1098,1135,1111,1083


the 30-event tier:


,tier,rows_in_data,planned,events_in_snapshot,in_census_too,reused_from_2600,reused_from_edge_cases,fetched_new,http_200,completed,with_lifi_line,destination_chains,sending_tokens,days
integrator,,,,,,,,,,,,,,
jumper.exchange,30 of 100+,20877,30,30,1,0,0,30,30,30,30,13,10,7
rabbykmsv2,30 of 100+,7758,30,30,0,1,0,29,30,29,30,8,11,7
metamask-bridge,30 of 100+,6853,30,30,0,1,0,29,30,29,26,8,15,7
base-app,30 of 100+,5765,30,30,0,1,0,29,30,26,27,9,8,7
_binancewallet,30 of 100+,5722,30,30,0,0,0,30,30,30,0,7,12,7
phantom,30 of 100+,5230,30,30,0,0,0,30,30,28,30,7,10,7
lifi-api,30 of 100+,3464,30,30,0,0,0,30,30,28,30,10,10,7
tangem,30 of 100+,2621,30,30,1,0,0,30,30,30,30,12,13,7
bitget,30 of 100+,2534,30,30,0,1,0,29,30,30,30,5,8,7


by design and source chain (query 144):


population_rows  events  events_per_1000_rows  distinct_hashes  reused_from_2600  reused_from_edge_cases  fetched_new  http_200  completed  integrators  amount_usd_provided
design            chain                                                                                                                                                                                   
census_swap_10k   all chains              751     751                1000.0              751                19                      13          719       751        741          NaN          41895314.76
                  ethereum                397     397                1000.0              397                11                       7          379       397        392         40.0          22484535.33
                  arbitrum                159     159                1000.0              159                 3                       6          150       159        157         19.0          13386049.24
                  bnb                      89      89                1000.0               89                 2                       0           87        89         89         20.0           3404842.47
                  base                     74      74                1000.0               74                 3                       0           71        74         72         18.0           2031528.74
                  polygon                  19      19                1000.0               19                 0                       0           19        19         18          7.0            275759.13
                  optimism                 13      13                1000.0               13                 0                       0           13        13         13          5.0            312599.85
integrator_strata all chains            84679    4165                  49.2             4163                79                      24         4062      4165       4086          NaN           6539488.56
                  base                  25623    1080                  42.1             1080                17                       1         1062      1080       1052        297.0            331265.52
                  ethereum              19865    1071                  53.9             1069                13                      13         1045      1071       1044        292.0           4654466.18
                  bnb                   18696     891                  47.7              891                17                       2          872       891        884        235.0            714507.91
                  arbitrum              11008     626                  56.9              626                14                       3          609       626        614        203.0            766848.09
                  polygon                5408     369                  68.2              369                11                       3          355       369        365        134.0             56907.28
                  optimism               4079     128                  31.4              128                 7                       2          119       128        127         68.0             15493.58
both              all chains            84679      48                   0.6               48                 0                       2           46        48         48          NaN           5521107.75
                  ethereum              19865      28                   1.4               28                 0                       2           26        28         28         16.0           4256540.77
                  arbitrum              11008      11                   1.0               11                 0                       0           11        11         11          7.0            597322.64
                  base                  25623       4                   0.2                4                 0                       0            4         4          4     

### 6.2 Status × substatus and the fee-line share per design (query `145_status_extension_status_and_fee_line_by_design`)

In [9]:
st = run("145_status_extension_status_and_fee_line_by_design")
display(st.set_index(["design", "status", "substatus"]))
tot = st[st["status"] == "all"].set_index("design")
for d in tot.index:
    r = tot.loc[d]
    print(f"{d}: {int(r['events']):,} events, with a quote {int(r['with_quote']):,}, with a LIFI-named line {int(r['with_lifi_line']):,} ({r['with_lifi_line'] / r['events']:.1%}), LI.FI fee > 0 on {int(r['lifi_fee_gt_0']):,} ({r['lifi_fee_gt_0'] / r['events']:.1%}), integrator fee > 0 on {int(r['integrator_fee_gt_0']):,} ({r['integrator_fee_gt_0'] / r['events']:.1%}), batchStatus on {int(r['with_batch_status']):,}")
print("the uniform 2,000's swap rows, for comparison (query 75): 1,363 rows, 1,210 with a LIFI-named line (88.8%), 1,007 with LI.FI fee > 0 (73.9%), 591 with an integrator fee > 0 (43.4%)")
# Does the response describe the selected event? The identity checks of 03 §7.3, over every row of the extension.
idc = run_sql("""SELECT count(*) AS rows_, count(*) FILTER (WHERE transaction_id_matches) AS transaction_id_matches, count(*) FILTER (WHERE sending_hash_matches) AS sending_hash_matches,
                        count(*) FILTER (WHERE coalesce(metadata_integrator, '') = coalesce(event_integrator, '')) AS integrator_matches, count(*) FILTER (WHERE tool = event_bridge) AS tool_equals_bridge,
                        count(*) FILTER (WHERE receiving_chainId <> event_destinationChainId) AS receiving_chain_differs, count(*) FILTER (WHERE amount_usd_provided IS NULL) AS no_provided_usd,
                        count(*) FILTER (WHERE in_census AND reused_from IS NOT NULL) AS census_reused, count(*) FILTER (WHERE NOT in_census AND reused_from IS NOT NULL) AS strata_only_reused
                 FROM ref_lifi_api_transfer_status_extension""")
print("identity over all rows (inline SQL on the loaded table):", idc.iloc[0].to_dict())
print("rows whose top-level transactionId is not the event's (bundles whose first call is another transfer, 03 §7.3), by integrator:", run_sql("SELECT coalesce(event_integrator, '(empty)') AS i, count(*) AS n FROM ref_lifi_api_transfer_status_extension WHERE NOT coalesce(transaction_id_matches, false) GROUP BY 1 ORDER BY n DESC").set_index("i")["n"].to_dict())


events  pct_of_design  http_200  with_quote  with_lifi_line  lifi_fee_gt_0  integrator_fee_gt_0  with_batch_status  amount_usd_provided
design            status substatus                                                                                                                                         
census_swap_10k   all    all           751         100.00       751         751             588            496                  268                 39          41895314.74
                  DONE   COMPLETED     741          98.67       741         741             578            488                  262                 37          41686129.86
                         PARTIAL         6           0.80         6           6               6              6                    2                  1            101660.63
                         REFUNDED        4           0.53         4           4               4              2                    4                  1            107524.25
integrator_strata all    all          4165         100.00      4165        4124            3812           3588                 1896                788           6539488.57
                  DONE   COMPLETED    4086          98.10      4086        4048            3738           3516                 1847                771           6495581.61
                         REFUNDED       59           1.42        59          57              55             53                   37                 13             14724.75
                         PARTIAL        20           0.48        20          19              19             19                   12                  4             29182.21
all               all    all          4868         100.00      4868        4827            4366           4052                 2140                827          42913695.55
                  DONE   COMPLETED    4779          98.17      4779        4741            4282           3972                 2085                808          42660603.71
                         REFUNDED       63           1.29        63          61              59             55                   41                 14            122249.00
                         PARTIAL        26           0.53        26          25              25             25                   14                  5            130842.84

census_swap_10k: 751 events, with a quote 751, with a LIFI-named line 588 (78.3%), LI.FI fee > 0 on 496 (66.0%), integrator fee > 0 on 268 (35.7%), batchStatus on 39
integrator_strata: 4,165 events, with a quote 4,124, with a LIFI-named line 3,812 (91.5%), LI.FI fee > 0 on 3,588 (86.1%), integrator fee > 0 on 1,896 (45.5%), batchStatus on 788
all: 4,868 events, with a quote 4,827, with a LIFI-named line 4,366 (89.7%), LI.FI fee > 0 on 4,052 (83.2%), integrator fee > 0 on 2,140 (44.0%), batchStatus on 827
the uniform 2,000's swap rows, for comparison (query 75): 1,363 rows, 1,210 with a LIFI-named line (88.8%), 1,007 with LI.FI fee > 0 (73.9%), 591 with an integrator fee > 0 (43.4%)


identity over all rows (inline SQL on the loaded table): {'rows_': 4868, 'transaction_id_matches': 4852, 'sending_hash_matches': 4868, 'integrator_matches': 4868, 'tool_equals_bridge': 4857, 'receiving_chain_differs': 137, 'no_provided_usd': 19, 'census_reused': 32, 'strata_only_reused': 101}
rows whose top-level transactionId is not the event's (bundles whose first call is another transfer, 03 §7.3), by integrator: {'request-network-api': 5, 'base-app': 3, 'request-api': 2, 'privacy-pools-v2': 2, 'smart-usdx': 1, 'lend-admin': 1, 'irisapp': 1, 'app_Treble': 1}


### 6.3 Distinct LI.FI rates and integrator rates per integrator (queries `146_status_extension_rates_by_integrator`, `148_status_extension_rate_mixtures`)

In [10]:
rates = run("146_status_extension_rates_by_integrator")
print(f"integrators in the snapshot: {len(rates)}; events {int(rates['events'].sum()):,}; with a LIFI-named line {int(rates['with_lifi_line'].sum()):,}; rows with more than one LIFI-named line (no rate computed): {int(rates['multi_line_rows'].sum())}")
with_line = rates[rates["with_lifi_line"] > 0]
print(f"integrators with at least one line: {len(with_line)}; of them with one LI.FI rate on their lines: {int((with_line['n_lifi_rates'] == 1).sum())}, two or more: {int((with_line['n_lifi_rates'] >= 2).sum())}; one integrator rate: {int((with_line['n_integrator_rates'] == 1).sum())}, two or more: {int((with_line['n_integrator_rates'] >= 2).sum())}")
two_plus = with_line[with_line["with_lifi_line"] >= 2]
print(f"integrators with two or more fee-carrying events: {len(two_plus)}; one LI.FI rate {int((two_plus['n_lifi_rates'] == 1).sum())}, two or more {int((two_plus['n_lifi_rates'] >= 2).sum())}; one integrator rate {int((two_plus['n_integrator_rates'] == 1).sum())}, two or more {int((two_plus['n_integrator_rates'] >= 2).sum())}")
print("integrators without any LIFI-named line, by events:", rates[rates["with_lifi_line"] == 0].sort_values("events", ascending=False).set_index("integrator")["events"].head(25).to_dict())
print("per integrator, the 30-event tier and the 10-event tier with 100 or more rows in the data... shown: every integrator with 100+ swap rows (rate:events pairs; 'none' = no LIFI-named line):")
show = ["rows_in_data", "events", "with_lifi_line", "n_lifi_rates", "lifi_rates_bps", "n_integrator_rates", "integrator_rates_bps", "line_kinds", "uniform_2000_lifi_rates"]
display(rates[rates["rows_in_data"] >= 100].set_index("integrator")[show])
print("integrators with 10 to 99 swap rows:")
display(rates[(rates["rows_in_data"] >= 10) & (rates["rows_in_data"] < 100)].set_index("integrator")[show])
print("the distinct LI.FI rates over the whole snapshot, by events (one row per event; 'none' = no line):")
lifi_rate_events = Counter()
for s in rates["lifi_rates_bps"]:
    for pair in str(s).split(", "):
        rate, n = pair.rsplit(":", 1)
        lifi_rate_events[rate] += int(n)
display(pd.Series(lifi_rate_events).sort_values(ascending=False).to_frame("events").T)


integrators in the snapshot: 584; events 4,868; with a LIFI-named line 4,366; rows with more than one LIFI-named line (no rate computed): 0
integrators with at least one line: 549; of them with one LI.FI rate on their lines: 509, two or more: 40; one integrator rate: 488, two or more: 61
integrators with two or more fee-carrying events: 407; one LI.FI rate 367, two or more 40; one integrator rate 346, two or more 61
integrators without any LIFI-named line, by events: {'Cow': 54, 'USDai': 47, '_binancewallet': 43, 'HashPort-Wallet-Prod': 30, '_chainless': 30, '_arbitrum': 13, 'jumperadvanced': 12, 'oku.trade': 11, 'backpack-wallet': 11, 'brave': 10, 'Pharos-Bridge': 10, 'prjx': 10, 'lifi-develop-api': 10, 'li.fi-playground': 10, 'tangem-pay-prod': 10, 'gmgntestenv': 10, 'Br56daGGRU26uhfJPN': 10, 'definitive-prod': 7, 'coin98-production': 6, 'lifi-local-api': 5, 'HashPort-Wallet-Dev': 4, 'lista.org': 4, 'Ethena': 3, 'dzap': 3, 'arc-polygon': 2}
per integrator, the 30-event tier and the 1

,rows_in_data,events,with_lifi_line,n_lifi_rates,lifi_rates_bps,n_integrator_rates,integrator_rates_bps,line_kinds,uniform_2000_lifi_rates
integrator,,,,,,,,,
jumper.exchange,20877,325,284,2,"2.0:242, 5.0:42, none:41",1,"0.0:284, none:41","Fixed:284, none:41","none:330, 2.0:257, 5.0:58"
rabbykmsv2,7758,94,94,1,0.0:94,2,"25.0:87, 12.0:7",Fixed:94,0.0:140
metamask-bridge,6853,64,35,1,"5.0:35, none:29",1,"0.0:35, none:29","Fixed:35, none:29","5.0:106, none:21, 4.0:1"
base-app,5765,31,27,2,"5.0:26, none:4, 4.5:1",2,"95.0:26, none:4, 85.5:1","Shared:27, none:4","5.0:75, none:7, 4.5:4"
_binancewallet,5722,43,0,0,none:43,0,none:43,none:43,none:229
phantom,5230,69,69,1,4.25:69,1,80.75:69,Shared:69,"4.25:89, 4.15:1, 4.2:1, 4.24:1"
lifi-api,3464,39,39,1,25.0:39,1,0.0:39,Fixed:39,25.0:46
tangem,2621,56,56,2,"20.0:34, 5.0:22",4,"0.0:21, 125.0:16, 144.0:10, 220.0:9",Fixed:56,"20.0:35, 5.0:4"
bitget,2534,30,30,1,0.0:30,1,30.0:30,Shared:30,0.0:41


integrators with 10 to 99 swap rows:


,rows_in_data,events,with_lifi_line,n_lifi_rates,lifi_rates_bps,n_integrator_rates,integrator_rates_bps,line_kinds,uniform_2000_lifi_rates
integrator,,,,,,,,,
gas-sweeper,98,10,10,1,25.0:10,1,0.0:10,Fixed:10,25.0:1
gmgntestenv,97,10,0,0,none:10,0,none:10,none:10,none:4
tangem-pay-prod,91,10,0,0,none:10,0,none:10,none:10,none:9
jumperadvanced,85,12,0,0,none:12,0,none:12,none:12,none:21
exa_app,77,10,10,1,25.0:10,1,25.0:10,Fixed:10,25.0:1
ambire-extension-prod,74,10,10,3,"7.5:8, 3.75:1, 7.0:1",3,"42.5:8, 21.25:1, 43.0:1",Shared:10,7.5:2
debot,72,10,10,1,25.0:10,1,0.0:10,Fixed:10,NaN
Curvy-Staging,68,10,10,1,25.0:10,1,0.0:10,Fixed:10,25.0:4
li.fi-playground,68,10,0,0,none:10,0,none:10,none:10,none:5


the distinct LI.FI rates over the whole snapshot, by events (one row per event; 'none' = no line):


,25.0,none,0.0,2.0,20.0,5.0,15.0,7.5,4.25,7.0,18.0,50.0,12.0,10.0,9.91,22.71,2.6,6.0,1.2,12.25,1.5,11.25,8.25,4.0,8.0,0.4,4.5,1.0,12.6,3.5,...,17.0,24.95,7.49,4.99,24.0,2.5,25.14,24.88,23.0,3.75,15.01,4.24,25.01,24.98,0.12,1.6,14.99,19.97,24.97,24.94,22.28,7.99,22.29,19.99,25.08,25.06,25.04,25.03,24.96,24.55
events,2695,502,314,267,206,154,123,71,69,55,42,39,38,29,29,29,16,14,12,12,11,10,10,10,9,8,8,7,7,6,...,2,2,2,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1


In [11]:
mix = run("148_status_extension_rate_mixtures")
print(f"integrators whose lines carry two or more LI.FI rates or two or more integrator rates: {mix['integrator'].nunique()}; rows (integrator × LI.FI rate × integrator rate × line kind): {len(mix)}")
display(mix.set_index(["integrator", "lifi_rate_bps", "integrator_rate_bps", "line_kind"]))


integrators whose lines carry two or more LI.FI rates or two or more integrator rates: 70; rows (integrator × LI.FI rate × integrator rate × line kind): 207


rows_in_data  events  in_census  destination_chains  sending_tokens  days  amount_usd_min  amount_usd_max
integrator          lifi_rate_bps integrator_rate_bps line_kind                                                                                                           
jumper.exchange     2.0           0.0                 Fixed             20877     242        218                  13              19     7            0.03       162819.13
                    5.0           0.0                 Fixed             20877      42         37                  14              11     7            0.43       165599.97
                    NaN           NaN                 none              20877      41         41                  11              10     7        10002.06       449993.16
rabbykmsv2          0.0           25.0                Fixed              7758      87         57                   8              15     7            0.10        99177.59
                                  12.0                Fixed              7758       7          7                   3               5     4       104808.91       299447.94
...                                                                       ...     ...        ...                 ...             ...   ...             ...             ...
cross-chain-feature 25.0          50.0                Fixed                 2       1          0                   1               1     1            0.19            0.19
luffyswap           25.0          0.0                 Fixed                 2       1          0                   1               1     1          342.54          342.54
                                  20.0                Fixed                 2       1          0                   1               1     1           22.17           22.17
onerootapp          25.0          0.0                 Fixed                 2       1          0                   1               1     1           12.49           12.49
                                  100.0               Fixed                 2       1          0                   1               1     1            1.97            1.97

[207 rows x 8 columns]

### 6.4 The census: quoted fee USD on the swap rows at $10,000 and above (query `147_status_extension_census_fee_usd`)

In [12]:
cen = run("147_status_extension_census_fee_usd")
t = cen[cen["integrator"] == "total"].iloc[0]
print(f"census: {int(t['events'])} events, completed {int(t['completed'])}, with a LIFI-named line {int(t['with_lifi_line'])} ({t['with_lifi_line'] / t['events']:.1%}), LI.FI fee > 0 on {int(t['lifi_fee_gt_0'])}, integrator fee > 0 on {int(t['integrator_fee_gt_0'])}; "
      f"headline USD {t['amount_usd']:,.2f}, API sending USD {t['sending_usd']:,.2f}; quoted LI.FI fee {t['lifi_fee_usd']:,.2f} ({t['lifi_bps']} bps of headline USD), integrator fee {t['integrator_fee_usd']:,.2f} ({t['integrator_bps']} bps), other lines {t['other_fee_usd']:,.2f} ({t['other_bps']} bps)")
print("by integrator (every integrator with a census event; the quoted fee, not a revenue estimate):")
display(cen.set_index("integrator"))
cen_rows = ext[ext["in_census"]].copy()
cen_rows["lifi_bps"] = 1e4 * cen_rows["fee_split_lifi_usd"].fillna(0) / cen_rows["amount_usd_provided"]
print("census events without a LIFI-named line, by integrator (events and provided-feed USD):")
display(cen_rows[cen_rows["quote_fee_lifi_usd"].fillna(0) <= 0].groupby(cen_rows["event_integrator"].fillna("(empty)")).agg(events=("tx_hash", "size"), usd_provided=("amount_usd_provided", "sum"), statuses=("substatus", lambda s: "|".join(f"{k}:{v}" for k, v in s.value_counts().items()))).sort_values("usd_provided", ascending=False).round(2))
print("census events by LI.FI fee USD, the ten largest (API USD and the quote's LIFI-named line):")
display(cen_rows.sort_values("fee_split_lifi_usd", ascending=False).head(10)[["chain", "block_time", "event_integrator", "event_bridge", "sending_token_symbol", "amount_usd_provided", "sending_amountUSD", "status", "substatus", "quote_fee_names", "quote_fee_lifi_percentages", "fee_split_lifi_usd", "fee_split_integrator_usd", "reused_from"]].round(2))
print("census events by headline USD band (provided-feed USD; the headline column is in query 147):")
bands = pd.cut(cen_rows["amount_usd_provided"], [10_000, 25_000, 50_000, 100_000, 250_000, 1_000_000, 1e12], right=False, labels=["$10k-25k", "$25k-50k", "$50k-100k", "$100k-250k", "$250k-1M", "$1M+"])
display(cen_rows.groupby(bands, observed=True).agg(events=("tx_hash", "size"), with_lifi_line=("quote_fee_lifi_usd", lambda s: int((s.fillna(0) > 0).sum())), lifi_fee_gt_0=("fee_split_lifi_usd", lambda s: int((s.fillna(0) > 0).sum())),
                                                    usd_provided=("amount_usd_provided", "sum"), lifi_fee_usd=("fee_split_lifi_usd", "sum"), integrator_fee_usd=("fee_split_integrator_usd", "sum")).round(2))


census: 751 events, completed 741, with a LIFI-named line 588 (78.3%), LI.FI fee > 0 on 496, integrator fee > 0 on 268; headline USD 41,895,314.75, API sending USD 41,960,667.12; quoted LI.FI fee 11,924.64 (2.846 bps of headline USD), integrator fee 42,245.00 (10.083 bps), other lines 13,608.29 (3.248 bps)
by integrator (every integrator with a census event; the quoted fee, not a revenue estimate):


,events,completed,with_lifi_line,lifi_fee_gt_0,integrator_fee_gt_0,amount_usd,sending_usd,lifi_fee_usd,integrator_fee_usd,other_fee_usd,lifi_bps,integrator_bps,other_bps
integrator,,,,,,,,,,,,,
total,751,741,588,496,268,41895314.75,41960667.12,11924.64,42245.00,13608.29,2.846,10.083,3.248
jumper.exchange,296,292,255,255,0,10897632.52,10903121.11,1663.44,0.00,2497.36,1.526,0.000,2.292
USDai,42,42,0,0,0,9572162.99,9568376.35,0.00,0.00,65.49,0.000,0.000,0.068
safe-app,16,16,16,16,16,4249685.10,4249875.95,1440.43,2675.09,21.12,3.390,6.295,0.050
mbridge-stable,3,3,3,3,3,2847465.39,2843487.50,142.17,142.17,7791.10,0.499,0.499,27.362
rabbykmsv2,64,62,64,0,64,2763067.59,2770513.06,0.00,5268.90,86.21,0.000,19.069,0.312
Cow,28,28,0,0,0,1686298.06,1691490.63,0.00,0.00,253.78,0.000,0.000,1.505
_ledger,34,34,34,34,34,1041036.98,1055366.88,2115.33,8073.49,534.21,20.319,77.552,5.132
kyberswap,21,21,21,0,21,918351.66,917616.87,0.00,695.72,199.53,0.000,7.576,2.173


census events without a LIFI-named line, by integrator (events and provided-feed USD):


,events,usd_provided,statuses
event_integrator,,,
USDai,42,9572162.99,COMPLETED:42
jumper.exchange,41,3878640.22,COMPLETED:41
Cow,28,1686298.06,COMPLETED:28
metamask-bridge,25,699146.20,COMPLETED:25
_binancewallet,13,439227.73,COMPLETED:13
lista.org,2,112685.11,COMPLETED:2
base-app,1,107686.98,COMPLETED:1
_arbitrum,4,77431.97,COMPLETED:4
jumperadvanced,2,27035.10,COMPLETED:2


census events by LI.FI fee USD, the ten largest (API USD and the quote's LIFI-named line):


,chain,block_time,event_integrator,event_bridge,sending_token_symbol,amount_usd_provided,sending_amountUSD,status,substatus,quote_fee_names,quote_fee_lifi_percentages,fee_split_lifi_usd,fee_split_integrator_usd,reused_from
3695,ethereum,2026-10-16T15:52:23,safe-app,glacis,USDG,1517757.14,1518048.66,DONE,COMPLETED,LIFI Shared Fee|Bridge Fee,0.0010,531.32,986.73,lifi_api_transfer_status_edge_cases_raw.jsonl
3214,ethereum,2026-10-14T12:04:11,trezor-suite,relaydepository,USDG,66291.50,68195.65,DONE,REFUNDED,LIFI Fixed Fee|Relayer Gas fee|Relayer Service fee,0.0280,340.98,1568.50,NaN
3627,ethereum,2026-10-16T11:11:59,soswap,near,ETH,112432.53,114072.64,DONE,COMPLETED,LIFI Fixed Fee|NearIntents Protocol Fee|1ClickSwap API Fee|NearIntents App Fee,0.0025,284.48,0.00,NaN
4250,ethereum,2026-10-19T14:16:47,coinapi-v2,layerswap,USDT,113329.79,113015.85,DONE,COMPLETED,LIFI Fixed Fee|LayerSwap fee,0.0025,282.54,0.00,NaN
447,arbitrum,2026-10-16T11:18:18,_ledger,relaydepository,ARB,139114.07,141499.98,DONE,COMPLETED,LIFI Fixed Fee|Relayer Gas fee|Relayer Service fee,0.0090,282.24,987.83,NaN
3711,ethereum,2026-10-16T17:23:59,safe-app,polymerStandard,USDC,999538.75,999871.90,DONE,COMPLETED,LIFI Shared Fee,0.0007,244.97,454.94,NaN
1577,base,2026-10-17T17:32:17,_ledger,across,USDC,118941.72,119249.31,DONE,COMPLETED,LIFI Fixed Fee|Relayer fee|Relayer gas fee,0.0090,238.50,834.75,NaN
3760,ethereum,2026-10-16T23:55:35,trezor-suite,layerswap,USDT,46048.60,47272.59,DONE,COMPLETED,LIFI Fixed Fee|LayerSwap fee,0.0280,236.77,1089.14,NaN
1566,base,2026-10-17T15:54:41,_ledger,across,USDC,99125.34,99661.53,DONE,COMPLETED,LIFI Fixed Fee|Relayer fee|Relayer gas fee,0.0090,199.32,697.63,NaN
3345,ethereum,2026-10-15T06:58:47,talisman,across,ETH,79017.27,79330.81,DONE,COMPLETED,LIFI Fixed Fee|Relayer fee|Relayer gas fee,0.0045,198.33,158.66,NaN


census events by headline USD band (provided-feed USD; the headline column is in query 147):


,events,with_lifi_line,lifi_fee_gt_0,usd_provided,lifi_fee_usd,integrator_fee_usd
amount_usd_provided,,,,,,
$10k-25k,428,367,312,6952433.71,4083.83,15147.92
$25k-50k,167,142,120,6088080.41,2629.91,8905.86
$50k-100k,83,47,42,5820684.29,2298.33,9332.32
$100k-250k,52,21,14,8327522.48,1702.57,6057.64
$250k-1M,15,8,5,6635686.75,556.98,1692.81
$1M+,6,3,3,8070907.11,653.03,1108.45


## 7. Snapshots written by this notebook, the manifest, and the key check

In [13]:
NEW_FILES = [EXT_CSV] + [p.name for p in NEW_JSONLS if p.exists()]
m = manifest_read()
display(pd.DataFrame.from_dict({k: m[k] for k in NEW_FILES}, orient="index")[["fetched_at_utc", "rows", "notebook_section", "source_url"]])
for k in NEW_FILES:
    print(f"{k}: {m[k].get('note')}")
counts_on_disk = {k: (len(pd.read_csv(REF / k, dtype=str)) if k.endswith(".csv") else sum(1 for _ in (REF / k).open())) for k in NEW_FILES}
sizes_mb = {k: round(os.path.getsize(REF / k) / 1e6, 1) for k in NEW_FILES}
print("rows on disk:", counts_on_disk, " manifest agrees:", all(counts_on_disk[k] == m[k]["rows"] for k in NEW_FILES))
print("file sizes (MB):", sizes_mb, " every raw file under 50 MB:", all(v < 50 for k, v in sizes_mb.items() if k.endswith(".jsonl")))
print("the committed files this notebook reads but never writes:", [PRIOR_2600, PRIOR_EDGE, "lifi_api_transfer_status_sample_2600.csv"])
if LIFI_KEY:
    leaked = [k for k in NEW_FILES if LIFI_KEY in (REF / k).read_text()]
    print("files under data/reference containing the key value:", leaked or "none")
    assert not leaked
print(f"calls this run: {RATE['calls']}, HTTP 429: {RATE['n429']}, network errors: {RATE['network_errors']}")


,fetched_at_utc,rows,notebook_section,source_url
lifi_api_transfer_status_extension.csv,2026-11-01T12:05:46+00:00,4868,07 §4,https://li.quest/v1/status?txHash=<tx_hash>
lifi_api_transfer_status_extension_census_raw.jsonl,2026-11-01T12:05:58+00:00,719,07 §4,https://li.quest/v1/status?txHash=<tx_hash>
lifi_api_transfer_status_extension_strata_ethereum_raw.jsonl,2026-11-01T12:39:40+00:00,1019,07 §4,https://li.quest/v1/status?txHash=<tx_hash>
lifi_api_transfer_status_extension_strata_arbitrum_raw.jsonl,2026-11-01T12:05:46+00:00,598,07 §4,https://li.quest/v1/status?txHash=<tx_hash>
lifi_api_transfer_status_extension_strata_base_raw.jsonl,2026-11-01T12:14:53+00:00,1058,07 §4,https://li.quest/v1/status?txHash=<tx_hash>
lifi_api_transfer_status_extension_strata_optimism_raw.jsonl,2026-11-01T12:54:46+00:00,118,07 §4,https://li.quest/v1/status?txHash=<tx_hash>
lifi_api_transfer_status_extension_strata_polygon_raw.jsonl,2026-11-01T12:56:31+00:00,355,07 §4,https://li.quest/v1/status?txHash=<tx_hash>
lifi_api_transfer_status_extension_strata_bnb_raw.jsonl,2026-11-01T12:28:45+00:00,868,07 §4,https://li.quest/v1/status?txHash=<tx_hash>


lifi_api_transfer_status_extension.csv: two designs over the 84,679 swap rows (has_source_swaps), seed 20261029: census_swap_10k = every row with headline amount_usd >= 10,000 (751 events); integrator_strata = per raw integrator string 30 events when it has 100+ swap rows, 10 when 10-99, every row under 10, a systematic draw over the rows sorted by (destination_chain_id, sending_asset_id, block_time) (4165 events over 584 strings; 48 events in both designs); 4868 rows, one per event; 133 rows reuse the committed record of their hash ({'lifi_api_transfer_status_sample_2600_raw.jsonl': 98, 'lifi_api_transfer_status_edge_cases_raw.jsonl': 35}) with its original fetch time; the rest from 4735 new calls: 4735 calls from 2026-11-01T12:05:46+00:00 to 2026-11-01T13:00:54+00:00 (86 per minute observed, 0.1 s sleep after each call and a wait until ratelimit-reset whenever ratelimit-remaining fell below 3: 7 such waits); final HTTP statuses {200: 4735}; HTTP 429 responses retried before a final r

rows on disk: {'lifi_api_transfer_status_extension.csv': 4868, 'lifi_api_transfer_status_extension_census_raw.jsonl': 719, 'lifi_api_transfer_status_extension_strata_ethereum_raw.jsonl': 1019, 'lifi_api_transfer_status_extension_strata_arbitrum_raw.jsonl': 598, 'lifi_api_transfer_status_extension_strata_base_raw.jsonl': 1058, 'lifi_api_transfer_status_extension_strata_optimism_raw.jsonl': 118, 'lifi_api_transfer_status_extension_strata_polygon_raw.jsonl': 355, 'lifi_api_transfer_status_extension_strata_bnb_raw.jsonl': 868}  manifest agrees: True
file sizes (MB): {'lifi_api_transfer_status_extension.csv': 6.1, 'lifi_api_transfer_status_extension_census_raw.jsonl': 9.9, 'lifi_api_transfer_status_extension_strata_ethereum_raw.jsonl': 14.7, 'lifi_api_transfer_status_extension_strata_arbitrum_raw.jsonl': 8.8, 'lifi_api_transfer_status_extension_strata_base_raw.jsonl': 15.7, 'lifi_api_transfer_status_extension_strata_optimism_raw.jsonl': 1.8, 'lifi_api_transfer_status_extension_strata_polygo

## 8. A third design: the census of bridge-only transfers at $100,000 and above

The two designs above sit on the swap rows, where the LI.FI fee lines occur (on the uniform 2,000, 0 of 831 transfers without source swaps carry a LIFI-named line). Those bridge-only rows hold most of the file's USD, and before this section only 25 of them above $100,000 had a status record. This design observes every transfer **without** `hasSourceSwaps` at $100,000 or more of headline USD (`census_bridge_100k`), so that every transfer in the file at $100k and above, swap or bridge-only, is observed rather than sampled. No draw; the selection is a fixed function of the data, as in §3.

The same helpers as §1 to §5: hashes already in the two notebook-03 JSONLs (or, were there any, in the seven raw files of §4) are reused with their original record and marked `reused_from`; one `/v1/status?txHash=<hash>` call per hash otherwise, paced as in §4 and recording `retries_429` and `window_waits` per call; the fetch log from the raw JSONL as in §5. The files are separate from the extension's: `lifi_api_transfer_status_extension_bridge_census_raw.jsonl` (raw) and `lifi_api_transfer_status_bridge_census.csv` (flattened, the same 77 columns in the same order as `lifi_api_transfer_status_extension.csv`; `sample_design = census_bridge_100k`, `in_census` and `in_strata` false, `stratum_*` empty, `reused_from` on the reused rows), loaded by `uv run lifi-build-db` as `ref_lifi_api_transfer_status_bridge_census`. Nothing written by §4 is modified.

### 8.1 Selection: every bridge-only row at headline USD ≥ $100,000, and the planned calls

In [14]:
BRIDGE_CENSUS_USD = 100_000
BRIDGE_DESIGN = "census_bridge_100k"
BRIDGE_CSV = "lifi_api_transfer_status_bridge_census.csv"
BRIDGE_JSONL = REF / "lifi_api_transfer_status_extension_bridge_census_raw.jsonl"
BANDS = ([100_000, 250_000, 1_000_000, 1e12], ["$100k-250k", "$250k-1M", "$1M+"])

br = run_sql(f"""
    SELECT chain, block_time, block_number, tx_hash, log_index, tx_from, transaction_id, bridge, integrator, referrer, sending_asset_id, receiver, min_amount,
           destination_chain_id, has_source_swaps, has_destination_call, amount_usd, amount_usd_provided, price_source, symbol
    FROM transfers_priced
    WHERE NOT has_source_swaps AND amount_usd >= {BRIDGE_CENSUS_USD}
    ORDER BY chain, block_time, log_index
""")
file_tot = run_sql("SELECT count(*) AS n, sum(amount_usd) AS usd, count(*) FILTER (WHERE NOT has_source_swaps) AS bridge_rows, sum(amount_usd) FILTER (WHERE NOT has_source_swaps) AS bridge_usd, "
                   "count(*) FILTER (WHERE amount_usd >= 100000) AS rows_100k, sum(amount_usd) FILTER (WHERE amount_usd >= 100000) AS usd_100k FROM transfers_priced").iloc[0]
br["integrator"] = br["integrator"].fillna("")
br["day"] = pd.to_datetime(br["block_time"]).dt.date
br["key"] = list(zip(br["chain"], br["tx_hash"], br["log_index"].astype(int)))
br["band"] = pd.cut(br["amount_usd"], BANDS[0], right=False, labels=BANDS[1])
assert br["key"].is_unique and not br["has_source_swaps"].any() and (br["amount_usd"] >= BRIDGE_CENSUS_USD).all()
br["in_census"], br["in_strata"] = False, False                   # design membership of the two swap-row designs: none
br["stratum_integrator"], br["stratum_rows_in_data"], br["stratum_planned"] = None, None, None
br["sample_design"] = BRIDGE_DESIGN
# Reuse: the two notebook-03 JSONLs (`prior`, §2) first, then the seven raw files of §4, so that no hash is fetched twice anywhere in this notebook.
prior_all = dict(prior)
for p in NEW_JSONLS:
    for r in jsonl_read(p):
        prior_all.setdefault(r["tx_hash"], (p.name, r))
br["reused_from"] = br["tx_hash"].map(lambda h: prior_all[h][0] if h in prior_all else None)
bridge_hashes_to_fetch = br.loc[br["reused_from"].isna(), "tx_hash"].drop_duplicates()
print(f"bridge-only census: {len(br):,} rows (has_source_swaps false, headline USD >= {BRIDGE_CENSUS_USD:,}) of {int(file_tot['bridge_rows']):,} bridge-only rows and {int(file_tot['n']):,} events; "
      f"headline USD {br['amount_usd'].sum():,.2f} = {br['amount_usd'].sum() / file_tot['usd']:.1%} of the file's {file_tot['usd']:,.2f} and {br['amount_usd'].sum() / file_tot['bridge_usd']:.1%} of the bridge-only rows' {file_tot['bridge_usd']:,.2f}; "
      f"priced by CoinGecko only {int((br['price_source'] == 'coingecko').sum())}; without provided-feed USD {int(br['amount_usd_provided'].isna().sum())}; distinct hashes {br['tx_hash'].nunique():,}; "
      f"integrator strings {br['integrator'].nunique()}; bridges {br['bridge'].nunique()}")
print(f"every transfer in the file at >= $100k: {int(file_tot['rows_100k']):,} events, {file_tot['usd_100k']:,.2f} USD = {file_tot['usd_100k'] / file_tot['usd']:.1%} of the file; of them swap rows (all in census_swap_10k) {int(file_tot['rows_100k']) - len(br):,}, bridge-only (this design) {len(br):,}")
print(f"reused from committed snapshots: {int(br['reused_from'].notna().sum())} rows ({br['reused_from'].value_counts().to_dict()}); in the seven raw files of §4: {int(br['tx_hash'].isin({r['tx_hash'] for p in NEW_JSONLS for r in jsonl_read(p)}).sum())}; hashes to fetch {len(bridge_hashes_to_fetch):,} (~{len(bridge_hashes_to_fetch) / 85:.0f} min at 85 per minute)")


def bridge_plan_table(frame, by):
    t = frame.groupby(by, observed=True).agg(events=("tx_hash", "size"), distinct_hashes=("tx_hash", "nunique"), integrators=("integrator", "nunique"),
                                             reused_2600=("reused_from", lambda s: int((s == PRIOR_2600).sum())), reused_edge=("reused_from", lambda s: int((s == PRIOR_EDGE).sum())),
                                             to_fetch=("reused_from", lambda s: int(frame.loc[s.index][s.isna()]["tx_hash"].nunique())), headline_usd=("amount_usd", "sum"))
    t["share_of_file_usd_pct"] = 100 * t["headline_usd"] / file_tot["usd"]
    return t.round({"headline_usd": 2, "share_of_file_usd_pct": 2})


print("planned calls, in total and by USD band (headline amount_usd):")
display(pd.concat([bridge_plan_table(br, "band"), bridge_plan_table(br.assign(band="all"), "band")]))
print("by source chain:")
display(bridge_plan_table(br, "chain").sort_values("events", ascending=False))
print("by integrator string (every string with a row; the bridge strings its rows use):")
pi = bridge_plan_table(br, "integrator").drop(columns="integrators").sort_values("headline_usd", ascending=False)
pi.insert(0, "bridges", br.groupby("integrator")["bridge"].agg(lambda s: ", ".join(f"{k}:{v}" for k, v in s.value_counts().items())))
pi.insert(1, "bridge_only_rows_in_data", run_sql("SELECT coalesce(integrator, '') AS integrator, count(*) AS n FROM transfers_priced WHERE NOT has_source_swaps GROUP BY 1").set_index("integrator")["n"])
pi.index = pi.index.map(lambda s: "(empty)" if s == "" else s)
display(pi)

bridge-only census: 309 rows (has_source_swaps false, headline USD >= 100,000) of 40,179 bridge-only rows and 124,858 events; headline USD 145,463,135.63 = 47.2% of the file's 307,969,358.59 and 59.8% of the bridge-only rows' 243,435,172.89; priced by CoinGecko only 0; without provided-feed USD 0; distinct hashes 309; integrator strings 10; bridges 11
every transfer in the file at >= $100k: 382 events, 168,497,251.97 USD = 54.7% of the file; of them swap rows (all in census_swap_10k) 73, bridge-only (this design) 309


reused from committed snapshots: 25 rows ({'lifi_api_transfer_status_edge_cases_raw.jsonl': 15, 'lifi_api_transfer_status_sample_2600_raw.jsonl': 10}); in the seven raw files of §4: 0; hashes to fetch 284 (~3 min at 85 per minute)
planned calls, in total and by USD band (headline amount_usd):


,events,distinct_hashes,integrators,reused_2600,reused_edge,to_fetch,headline_usd,share_of_file_usd_pct
band,,,,,,,,
$100k-250k,183,183,9,4,0,179,2.686523e+07,8.72
$250k-1M,110,110,8,5,0,105,5.055339e+07,16.42
$1M+,16,16,6,1,15,0,6.804452e+07,22.09
all,309,309,10,10,15,284,1.454631e+08,47.23


by source chain:


,events,distinct_hashes,integrators,reused_2600,reused_edge,to_fetch,headline_usd,share_of_file_usd_pct
chain,,,,,,,,
ethereum,177,177,9,5,13,159,96634383.24,31.38
arbitrum,97,97,6,4,2,91,39307124.73,12.76
base,27,27,4,1,0,26,6865285.09,2.23
optimism,4,4,1,0,0,4,1444839.14,0.47
polygon,3,3,1,0,0,3,1075222.40,0.35
bnb,1,1,1,0,0,1,136281.04,0.04


by integrator string (every string with a row; the bridge strings its rows use):


,bridges,bridge_only_rows_in_data,events,distinct_hashes,reused_2600,reused_edge,to_fetch,headline_usd,share_of_file_usd_pct
integrator,,,,,,,,,
_arbitrum,"polymerStandard:30, across:7, glacis:5, relaydepository:3, lifiIntents:3, mayan:3",1644,51,51,3,4,44,54248220.64,17.61
jumper.exchange,"polymerStandard:57, relaydepository:38, lifiIntents:22, glacis:16, near:10, celercircl...",20031,160,160,6,2,152,44234794.46,14.36
Pharos-Bridge,polymerStandard:3,24,3,3,0,2,1,16735953.88,5.43
_binancewallet,"polymerStandard:41, glacis:1",9387,42,42,1,3,38,14240986.48,4.62
Cow,"relaydepository:21, near:6, mayan:2",2075,29,29,0,0,29,6288283.04,2.04
jumperadvanced,"polymerStandard:11, stargateV2:1, across:1, glacis:1",1034,14,14,0,1,13,4635026.04,1.51
oku.trade,glacis:5,319,5,5,0,3,2,4102465.24,1.33
derive-v3,polymerStandard:3,534,3,3,0,0,3,475072.16,0.15
os-prod,layerswap:1,1369,1,1,0,0,1,253006.28,0.08


### 8.2 Fetch: one `/v1/status?txHash=<hash>` call per hash not already in a committed snapshot

The loop of §4 with the raw file as a parameter: calls go to `lifi_api_transfer_status_extension_bridge_census_raw.jsonl`, each record carrying the design, the event keys, `request_params` (no key), the fetch time, the HTTP status, `ratelimit_remaining`, `retries_429`, `window_waits` and the raw response; the loop appends as responses arrive and skips hashes already in the file, so a run that stops resumes. The flattened CSV is written once every hash has a record, reused rows taking the committed record with its original fetch time and `reused_from` naming the file. Re-executed with the files present, this cell reads them and makes no call.

In [15]:
def planned_bridge_calls():
    "One call per hash to fetch, in selection order (chain, time)."
    calls, seen = [], set()
    for ev in br[br["reused_from"].isna()].itertuples(index=False):
        if ev.tx_hash in seen:
            continue
        seen.add(ev.tx_hash)
        calls.append({"path": BRIDGE_JSONL, "chain": ev.chain, "tx_hash": ev.tx_hash, "log_index": int(ev.log_index), "event_transactionId": ev.transaction_id,
                      "sample_design": BRIDGE_DESIGN, "in_census": False, "in_strata": False})
    return calls


def fetch_missing_into(calls, jsonls):
    "§4's fetch_missing with the raw files as a parameter: skips hashes already in any of `jsonls`, appends each response to the call's path as it arrives."
    done = {r["tx_hash"]: r for p in jsonls for r in jsonl_read(p)}
    todo = [c for c in calls if c["tx_hash"] not in done]
    print(f"{len(calls):,} calls planned, {len(done):,} already in the raw file, {len(todo):,} to fetch (~{len(todo) / 85:.0f} min at 85 per minute)")
    t_start, handles = time.time(), {}
    try:
        for i, c in enumerate(todo):
            status, body, remaining, meta = get_json(f"{LIFI}/status", params={"txHash": c["tx_hash"]})
            rec = {"lookup": "tx_hash", "chain": c["chain"], "lookup_value": c["tx_hash"], "bridge_param": None, "request_params": {"txHash": c["tx_hash"]},
                   "tx_hash": c["tx_hash"], "log_index": c["log_index"], "event_transactionId": c["event_transactionId"], "sample_design": c["sample_design"], "in_census": c["in_census"], "in_strata": c["in_strata"],
                   "fetched_at_utc": now_utc(), "http_status": status, "ratelimit_remaining": remaining, "retries_429": meta["retries_429"], "window_waits": meta["window_waits"], "response": body}
            fh = handles.get(c["path"])
            if fh is None:
                fh = handles[c["path"]] = c["path"].open("a")
            fh.write(json.dumps(rec) + "\n")
            fh.flush()
            done[c["tx_hash"]] = rec
            if (i + 1) % 100 == 0 or i + 1 == len(todo):
                print(f"  {i + 1}/{len(todo)} fetched, {(time.time() - t_start) / 60:.1f} min, HTTP 429 so far {RATE['n429']}, window waits {RATE['window_waits']} ({RATE['window_wait_s']:.0f} s), min ratelimit-remaining seen {RATE['min_remaining']}")
    finally:
        for fh in handles.values():
            fh.close()
    return done


def fetch_bridge_census():
    done = fetch_missing_into(planned_bridge_calls(), [BRIDGE_JSONL])
    missing = [h for h in br["tx_hash"].unique() if h not in done and h not in prior_all]
    assert not missing, f"{len(missing)} hashes without a record"
    rows = []
    for ev in br.to_dict("records"):
        name, rec = prior_all[ev["tx_hash"]] if ev["tx_hash"] in prior_all else (None, done[ev["tx_hash"]])
        row = flatten_status(ev, rec)
        row.update({"sample_design": BRIDGE_DESIGN, "lookup": "tx_hash", "lookup_value": ev["tx_hash"], "in_census": False, "in_strata": False,
                    "stratum_integrator": None, "stratum_rows_in_data": None, "stratum_planned": None, "reused_from": name})
        rows.append(row)
    out = pd.DataFrame(rows)
    ext_cols = list(pd.read_csv(REF / EXT_CSV, nrows=0).columns)   # the 77 columns of the extension CSV, in its order
    assert set(ext_cols) == set(out.columns), (sorted(set(ext_cols) - set(out.columns)), sorted(set(out.columns) - set(ext_cols)))
    return out[ext_cols]


def bridge_note():
    recs = jsonl_read(BRIDGE_JSONL)
    return (f"third design: census_bridge_100k = every row without has_source_swaps with headline amount_usd >= {BRIDGE_CENSUS_USD:,} ({len(br)} events, {br['amount_usd'].sum():,.2f} USD, "
            f"{br['amount_usd'].sum() / file_tot['usd']:.1%} of the file's headline USD); no draw; {len(br)} rows, one per event, the 77 columns of lifi_api_transfer_status_extension.csv in its order "
            f"(sample_design = census_bridge_100k, in_census and in_strata false, stratum_* empty); {int(br['reused_from'].notna().sum())} rows reuse the committed record of their hash "
            f"({br['reused_from'].value_counts().to_dict()}) with its original fetch time; the rest from {len(recs)} new calls: {fetch_span(recs)}; raw responses in {BRIDGE_JSONL.name}")


calls_before = RATE["calls"]
brc = snapshot_csv(BRIDGE_CSV, fetch_bridge_census, "07 §8.2", STATUS_URL, note=bridge_note, fetched_at=lambda: min(r["fetched_at_utc"] for r in jsonl_read(BRIDGE_JSONL)))
brc = read_status_csv(BRIDGE_CSV)
if BRIDGE_JSONL.exists() and BRIDGE_JSONL.name not in manifest_read():
    recs = jsonl_read(BRIDGE_JSONL)
    manifest_write(BRIDGE_JSONL.name, STATUS_URL, len(recs), "07 §8.2", min(r["fetched_at_utc"] for r in recs),
                   note=(f"raw JSON responses, one line per call (the bridge-only census hashes not already in a committed snapshot), with sample_design, in_census, in_strata, the event keys, "
                         f"request parameters (no key), HTTP status, ratelimit-remaining, retries_429 and window_waits; {fetch_span(recs)}"))
ext_cols = list(pd.read_csv(REF / EXT_CSV, nrows=0).columns)
print(f"{BRIDGE_CSV}: {brc.shape}; columns equal to the extension CSV's in order: {list(brc.columns) == ext_cols}; sample_design {brc['sample_design'].unique().tolist()}; "
      f"in_census any {bool(brc['in_census'].any())}, in_strata any {bool(brc['in_strata'].any())}; stratum_* all empty {bool(brc[['stratum_integrator', 'stratum_rows_in_data', 'stratum_planned']].isna().all().all())}; "
      f"reused_from {brc['reused_from'].value_counts(dropna=False).to_dict()}; HTTP statuses {brc['http_status'].value_counts().to_dict()}")
assert len(brc) == len(br) and set(zip(brc["chain"], brc["tx_hash"], brc["log_index"].astype(int))) == set(br["key"])
assert brc["tx_hash"].is_unique, "one record per hash: no hash fetched twice"
new_hashes = {r["tx_hash"] for r in jsonl_read(BRIDGE_JSONL)}
assert not (new_hashes & set(prior_all)), "a hash fetched here is in no earlier snapshot"
print(f"calls this run: {RATE['calls']}, HTTP 429: {RATE['n429']}, network errors: {RATE['network_errors']}, window waits: {RATE['window_waits']}, min ratelimit-remaining seen: {RATE['min_remaining']} (this section's calls: {RATE['calls'] - calls_before})")

lifi_api_transfer_status_bridge_census.csv: read snapshot, 309 rows, fetched 2026-11-01T13:56:43+00:00 (no network call)
lifi_api_transfer_status_bridge_census.csv: (309, 77); columns equal to the extension CSV's in order: True; sample_design ['census_bridge_100k']; in_census any False, in_strata any False; stratum_* all empty True; reused_from {nan: 284, 'lifi_api_transfer_status_edge_cases_raw.jsonl': 15, 'lifi_api_transfer_status_sample_2600_raw.jsonl': 10}; HTTP statuses {np.int64(200): 309}
calls this run: 0, HTTP 429: 0, network errors: 0, window waits: 0, min ratelimit-remaining seen: None (this section's calls: 0)


### 8.3 The fetch log of the bridge-only census, from its raw JSONL

The same columns as §5, for `lifi_api_transfer_status_extension_bridge_census_raw.jsonl` alone; the reused records are not calls of this notebook and are not in this table.

In [16]:
bridge_recs = jsonl_read(BRIDGE_JSONL)
assert bridge_recs, "no raw records: the fetch of §8.2 did not run"
display(pd.DataFrame({"bridge census (extension_bridge_census_raw)": fetch_log(bridge_recs)}))
rem_b = pd.Series([int(r["ratelimit_remaining"]) for r in bridge_recs if str(r.get("ratelimit_remaining") or "").isdigit()])
print("ratelimit-remaining over the calls:", rem_b.describe(percentiles=[0.01, 0.05, 0.1, 0.5]).round(1).to_dict(), "; calls with ratelimit-remaining below", REMAINING_FLOOR, ":", int((rem_b < REMAINING_FLOOR).sum()))
ts_b = pd.Series(sorted(pd.Timestamp(r["fetched_at_utc"]) for r in bridge_recs))
per_min_b = ts_b.dt.floor("min").value_counts().sort_index()
print(f"calls per clock minute: max {int(per_min_b.max())}, median {per_min_b.median():.0f}, minutes with calls {len(per_min_b)}; minutes with more than 100 calls: {int((per_min_b > 100).sum())}")
print(f"runs (gaps above 60 s between consecutive calls split them): {int((ts_b.diff().dt.total_seconds() > 60).sum()) + 1}; gaps above 60 s: {[round(g) for g in ts_b.diff().dt.total_seconds().dropna() if g > 60]}")
print("records by design and HTTP status:", pd.crosstab([r["sample_design"] for r in bridge_recs], [r["http_status"] for r in bridge_recs]).to_dict())
print("records whose response is not a dict (an error body):", sum(1 for r in bridge_recs if not isinstance(r.get("response"), dict)))
print("hashes in this file also in any other raw JSONL under data/reference:", len({r["tx_hash"] for r in bridge_recs} & set(prior_all)))

,bridge census (extension_bridge_census_raw)
calls,284
http_statuses,{200: 284}
first_call_utc,2026-11-01T13:56:43+00:00
last_call_utc,2026-11-01T13:59:30+00:00
span_minutes,2.8
calls_per_minute,102.0
min_ratelimit_remaining,2
max_gap_between_calls_s,12.0
http_429_retried,0
window_waits,2


ratelimit-remaining over the calls: {'count': 284.0, 'mean': 52.0, 'std': 27.6, 'min': 2.0, '1%': 3.0, '5%': 9.0, '10%': 14.3, '50%': 52.0, 'max': 99.0} ; calls with ratelimit-remaining below 3 : 2
calls per clock minute: max 101, median 76, minutes with calls 4; minutes with more than 100 calls: 1
runs (gaps above 60 s between consecutive calls split them): 1; gaps above 60 s: []
records by design and HTTP status: {200: {'census_bridge_100k': 284}}
records whose response is not a dict (an error body): 0
hashes in this file also in any other raw JSONL under data/reference: 0


### 8.4 Results: status, fee lines and quoted fee USD on the bridge-only census, and every transfer at $100k and above (queries `149_status_100k_all_transfers_fee_line_by_band`, `150_status_bridge_census_fee_usd_by_integrator`, `151_status_bridge_census_status_and_band`)

Read from `ref_lifi_api_transfer_status_bridge_census` (and, for the combined table, `ref_lifi_api_transfer_status_extension` for the swap rows and `ref_lifi_api_transfer_status_edge_cases` as the fallback that shows nothing is missing). If the CSV was just written in §8.2 the table does not exist yet and the database is rebuilt here once. "Fee line" is any entry of the quote's `feeCosts` (`quote_fee_names` non-empty); "LIFI-named line" is one whose name starts with `LIFI`; the fee USD columns are query 147's. Descriptive only: no weighting, no schedule, no revenue figure.

In [17]:
with duckdb.connect(str(paths.db_path()), read_only=True) as con:
    have_bridge = "ref_lifi_api_transfer_status_bridge_census" in {t[0] for t in con.execute("SHOW TABLES").fetchall()}
if not have_bridge:
    print("the database has no ref_lifi_api_transfer_status_bridge_census table yet: rebuilding analysis/lifi.duckdb from the CSVs and the snapshots")
    build(out=None)
n_brc = int(run_sql("SELECT count(*) AS n FROM ref_lifi_api_transfer_status_bridge_census")["n"].iloc[0])
assert n_brc == len(brc), (n_brc, len(brc))
print(f"ref_lifi_api_transfer_status_bridge_census: {n_brc:,} rows, equal to the CSV")

# Fee-line presence, exactly: any line, LIFI-named lines, fee USD, batchStatus; and the identity checks of §6.2 over the bridge-only census.
fl = run_sql("""SELECT count(*) AS rows_, count(*) FILTER (WHERE http_status = 200) AS http_200, count(*) FILTER (WHERE quote_tool IS NOT NULL) AS with_quote,
                       count(*) FILTER (WHERE coalesce(CAST(quote_fee_names AS VARCHAR), '') <> '') AS with_any_fee_line,
                       count(*) FILTER (WHERE coalesce(quote_fee_lifi_usd, 0) > 0) AS with_lifi_line,
                       count(*) FILTER (WHERE CAST(quote_fee_names AS VARCHAR) LIKE '%LIFI%') AS with_lifi_named_line_by_name,
                       count(*) FILTER (WHERE coalesce(fee_split_lifi_usd, 0) > 0) AS lifi_fee_gt_0, count(*) FILTER (WHERE coalesce(fee_split_integrator_usd, 0) > 0) AS integrator_fee_gt_0,
                       count(*) FILTER (WHERE coalesce(quote_fee_other_usd, 0) > 0) AS other_fee_gt_0, round(sum(quote_fee_usd), 2) AS quote_fee_usd_total, round(sum(quote_fee_lifi_usd), 2) AS quote_fee_lifi_usd_total,
                       round(sum(quote_fee_other_usd), 2) AS quote_fee_other_usd_total, count(*) FILTER (WHERE batch_n IS NOT NULL) AS with_batch_status, round(sum(quote_gas_usd), 2) AS quote_gas_usd_total,
                       count(*) FILTER (WHERE transaction_id_matches) AS transaction_id_matches, count(*) FILTER (WHERE sending_hash_matches) AS sending_hash_matches,
                       count(*) FILTER (WHERE coalesce(metadata_integrator, '') = coalesce(event_integrator, '')) AS integrator_matches, count(*) FILTER (WHERE tool = event_bridge) AS tool_equals_bridge,
                       count(*) FILTER (WHERE receiving_chainId <> event_destinationChainId) AS receiving_chain_differs, count(*) FILTER (WHERE amount_usd_provided IS NULL) AS no_provided_usd,
                       count(*) FILTER (WHERE reused_from IS NOT NULL) AS reused
                FROM ref_lifi_api_transfer_status_bridge_census""")
print("fee-line presence and identity over the bridge-only census (inline SQL on the loaded table):", fl.iloc[0].to_dict())
print("fee line names present, by rows (quote_fee_names; every line of every quote):", run_sql("SELECT coalesce(CAST(quote_fee_names AS VARCHAR), '(no line)') AS names, count(*) AS n FROM ref_lifi_api_transfer_status_bridge_census GROUP BY 1 ORDER BY n DESC").set_index("names")["n"].to_dict())
print("rows with a fee line of any kind (name, tool, integrator, status, fee USD):")
display(run_sql("SELECT chain, event_integrator, event_bridge, sending_token_symbol, round(amount_usd_provided, 2) AS amount_usd_provided, status, substatus, quote_fee_names, quote_fee_usd, quote_fee_lifi_usd, quote_fee_other_usd, fee_split_lifi_usd, fee_split_integrator_usd FROM ref_lifi_api_transfer_status_bridge_census WHERE coalesce(CAST(quote_fee_names AS VARCHAR), '') <> '' ORDER BY quote_fee_usd DESC NULLS LAST"))
print("rows whose top-level transactionId is not the event's (bundles whose first call is another transfer, 03 §7.3), by integrator:", run_sql("SELECT coalesce(event_integrator, '(empty)') AS i, count(*) AS n FROM ref_lifi_api_transfer_status_bridge_census WHERE NOT coalesce(transaction_id_matches, false) GROUP BY 1 ORDER BY n DESC").set_index("i")["n"].to_dict())

print("\nstatus × substatus and the USD bands (query 151):")
st_b = run("151_status_bridge_census_status_and_band")
display(st_b.set_index(["status", "substatus"]))
print("\nby bridge (every bridge-only census row; other-line USD is the quote's non-LIFI lines):")
display(run_sql("SELECT event_bridge AS bridge, count(*) AS events, count(*) FILTER (WHERE coalesce(CAST(quote_fee_names AS VARCHAR), '') <> '') AS with_any_fee_line, count(*) FILTER (WHERE coalesce(quote_fee_lifi_usd, 0) > 0) AS with_lifi_line, round(sum(amount_usd_provided), 2) AS amount_usd_provided, round(sum(quote_fee_other_usd), 2) AS other_fee_usd, string_agg(DISTINCT CAST(quote_fee_names AS VARCHAR), ' | ') AS fee_line_names FROM ref_lifi_api_transfer_status_bridge_census GROUP BY 1 ORDER BY amount_usd_provided DESC"))
print("\nby integrator (query 150; the quoted fee, not a revenue estimate):")
display(run("150_status_bridge_census_fee_usd_by_integrator").set_index("integrator"))
print("\nrows with batchStatus, by integrator and bridge (bundles):")
display(run_sql("SELECT coalesce(event_integrator, '(empty)') AS integrator, event_bridge, count(*) AS rows_, sum(CAST(batch_n AS BIGINT)) AS batch_entries, sum(CAST(batch_calls_n AS BIGINT)) AS batch_calls, string_agg(DISTINCT batch_call_tools, ' | ') AS batch_call_tools FROM ref_lifi_api_transfer_status_bridge_census WHERE batch_n IS NOT NULL GROUP BY 1, 2 ORDER BY rows_ DESC"))

ref_lifi_api_transfer_status_bridge_census: 309 rows, equal to the CSV


fee-line presence and identity over the bridge-only census (inline SQL on the loaded table): {'rows_': 309.0, 'http_200': 309.0, 'with_quote': 309.0, 'with_any_fee_line': 133.0, 'with_lifi_line': 0.0, 'with_lifi_named_line_by_name': 0.0, 'lifi_fee_gt_0': 0.0, 'integrator_fee_gt_0': 0.0, 'other_fee_gt_0': 133.0, 'quote_fee_usd_total': 9585.41, 'quote_fee_lifi_usd_total': 0.0, 'quote_fee_other_usd_total': 9585.41, 'with_batch_status': 0.0, 'quote_gas_usd_total': 16.69, 'transaction_id_matches': 309.0, 'sending_hash_matches': 309.0, 'integrator_matches': 309.0, 'tool_equals_bridge': 309.0, 'receiving_chain_differs': 13.0, 'no_provided_usd': 0.0, 'reused': 25.0}


fee line names present, by rows (quote_fee_names; every line of every quote): {'(no line)': 176, 'Relayer Gas fee|Relayer Service fee': 58, 'Bridge Fee': 27, 'NearIntents Protocol Fee|1ClickSwap API Fee|NearIntents App Fee': 16, 'Relayer fee|Relayer gas fee': 12, 'LayerSwap fee': 6, 'Transaction Fee': 5, 'Relayer Service fee': 4, 'Polymer fee|CCTP service fee': 2, 'Relayer fee|Relayer gas fee|LP fee': 1, 'Bridge Fee|Airlift Fee': 1, 'LayerZero native fee': 1}
rows with a fee line of any kind (name, tool, integrator, status, fee USD):


,chain,event_integrator,event_bridge,sending_token_symbol,amount_usd_provided,status,substatus,quote_fee_names,quote_fee_usd,quote_fee_lifi_usd,quote_fee_other_usd,fee_split_lifi_usd,fee_split_integrator_usd
0,arbitrum,Cow,near,USDC,442091.30,DONE,COMPLETED,NearIntents Protocol Fee|1ClickSwap API Fee|NearIntents App Fee,531.095700,0.0,531.095700,NaN,NaN
1,arbitrum,Cow,near,USDC,441809.22,DONE,COMPLETED,NearIntents Protocol Fee|1ClickSwap API Fee|NearIntents App Fee,530.482500,0.0,530.482500,NaN,NaN
2,arbitrum,Cow,near,USDC,426439.76,DONE,COMPLETED,NearIntents Protocol Fee|1ClickSwap API Fee|NearIntents App Fee,512.002800,0.0,512.002800,NaN,NaN
3,arbitrum,Cow,near,USDC,358872.04,DONE,COMPLETED,NearIntents Protocol Fee|1ClickSwap API Fee|NearIntents App Fee,430.738000,0.0,430.738000,NaN,NaN
4,ethereum,jumper.exchange,layerswap,DAI,368405.62,DONE,COMPLETED,LayerSwap fee,346.549461,0.0,346.549461,NaN,NaN
5,ethereum,jumper.exchange,layerswap,DAI,360332.74,DONE,COMPLETED,LayerSwap fee,339.464255,0.0,339.464255,NaN,NaN
6,arbitrum,Cow,near,USDC,763446.91,DONE,COMPLETED,NearIntents Protocol Fee|1ClickSwap API Fee|NearIntents App Fee,305.224800,0.0,305.224800,NaN,NaN
7,ethereum,jumper.exchange,layerswap,USDC,296177.80,DONE,COMPLETED,LayerSwap fee,284.373262,0.0,284.373262,NaN,NaN
8,ethereum,Cow,relaydepository,ETH,250789.33,DONE,COMPLETED,Relayer Gas fee|Relayer Service fee,275.845200,0.0,275.845200,NaN,NaN
9,ethereum,Cow,relaydepository,ETH,250337.92,DONE,COMPLETED,Relayer Gas fee|Relayer Service fee,275.611700,0.0,275.611700,NaN,NaN


rows whose top-level transactionId is not the event's (bundles whose first call is another transfer, 03 §7.3), by integrator: {}

status × substatus and the USD bands (query 151):


events  pct_of_census  band_100k_250k  band_250k_1m  band_1m_plus  http_200  reused  with_quote  with_any_fee_line  with_lifi_line  lifi_fee_gt_0  integrator_fee_gt_0  with_batch_status    amount_usd
status substatus                                                                                                                                                                                                         
all    all           309         100.00             183           110            16       309      25         309                133               0              0                    0                  0  1.454631e+08
DONE   COMPLETED     308          99.68             183           109            16       308      24         308                133               0              0                    0                  0  1.449853e+08
       REFUNDED        1           0.32               0             1             0         1       1           1                  0               0              0                    0                  0  4.778687e+05

... (2 outputs left out)


by integrator (query 150; the quoted fee, not a revenue estimate):


,events,bridges,bridge_names,http_200,reused,completed,with_quote,with_any_fee_line,with_lifi_line,lifi_fee_gt_0,integrator_fee_gt_0,other_fee_gt_0,with_batch_status,fee_line_names,amount_usd,sending_usd,lifi_fee_usd,integrator_fee_usd,other_fee_usd,lifi_bps,integrator_bps,other_bps
integrator,,,,,,,,,,,,,,,,,,,,,,
total,309,11,"across, celercircle, glacis, layerswap, lifiIntents, mayan, near, polymer, polymerStan...",309,25,308,309,133,0,0,0,133,0,Polymer fee|CCTP service fee | Transaction Fee | LayerSwap fee | Bridge Fee | Relayer ...,1.454631e+08,1.454419e+08,0.0,0.0,9585.41,0.0,0.0,0.659
_arbitrum,51,6,"across, glacis, lifiIntents, mayan, polymerStandard, relaydepository",51,7,51,51,15,0,0,0,15,0,Relayer fee|Relayer gas fee|LP fee | Bridge Fee | Relayer fee|Relayer gas fee | Relaye...,5.424822e+07,5.425973e+07,0.0,0.0,307.94,0.0,0.0,0.057
jumper.exchange,160,10,"across, celercircle, glacis, layerswap, lifiIntents, mayan, near, polymer, polymerStan...",160,8,159,160,80,0,0,0,80,0,Relayer fee|Relayer gas fee | Relayer Gas fee|Relayer Service fee | Transaction Fee | ...,4.423479e+07,4.421969e+07,0.0,0.0,3712.11,0.0,0.0,0.839
Pharos-Bridge,3,1,polymerStandard,3,2,3,3,0,0,0,0,0,0,NaN,1.673595e+07,1.672807e+07,0.0,0.0,0.00,0.0,0.0,0.000
_binancewallet,42,2,"glacis, polymerStandard",42,4,42,42,1,0,0,0,1,0,Bridge Fee|Airlift Fee,1.424099e+07,1.423572e+07,0.0,0.0,40.79,0.0,0.0,0.029
Cow,29,3,"mayan, near, relaydepository",29,0,29,29,27,0,0,0,27,0,Relayer Gas fee|Relayer Service fee | NearIntents Protocol Fee|1ClickSwap API Fee|Near...,6.288283e+06,6.287941e+06,0.0,0.0,5422.12,0.0,0.0,8.623
jumperadvanced,14,4,"across, glacis, polymerStandard, stargateV2",14,1,14,14,3,0,0,0,3,0,Bridge Fee | Relayer fee|Relayer gas fee | LayerZero native fee,4.635026e+06,4.629892e+06,0.0,0.0,31.70,0.0,0.0,0.068
oku.trade,5,1,glacis,5,3,5,5,5,0,0,0,5,0,Bridge Fee,4.102465e+06,4.102659e+06,0.0,0.0,4.50,0.0,0.0,0.011
derive-v3,3,1,polymerStandard,3,0,3,3,0,0,0,0,0,0,NaN,4.750722e+05,4.756645e+05,0.0,0.0,0.00,0.0,0.0,0.000



rows with batchStatus, by integrator and bridge (bundles):


,integrator,event_bridge,rows_,batch_entries,batch_calls,batch_call_tools


In [18]:
# Every transfer in the file at $100k and above, swap and bridge-only (query 149): the record comes from the extension's census for swap rows, from the bridge-only census for the others, else from the edge cases.
every = run("149_status_100k_all_transfers_fee_line_by_band")
display(every.set_index(["route_kind", "usd_band"]))
tot_all = every[(every["route_kind"] == "all") & (every["usd_band"] == "all")].iloc[0]
tot_sw = every[(every["route_kind"] == "swap") & (every["usd_band"] == "all")].iloc[0]
tot_br = every[(every["route_kind"] == "bridge_only") & (every["usd_band"] == "all")].iloc[0]
print(f"every transfer at >= $100k: {int(tot_all['events']):,} events, {tot_all['amount_usd']:,.2f} USD ({tot_all['pct_of_file_usd']}% of the file); with a status record {int(tot_all['with_record']):,} (extension {int(tot_all['record_from_extension'])}, bridge census {int(tot_all['record_from_bridge_census'])}, edge cases {int(tot_all['record_from_edge_cases'])}), without {int(tot_all['without_record'])}; "
      f"completed {int(tot_all['completed']):,}; with a LIFI-named line {int(tot_all['with_lifi_line'])} ({tot_all['pct_with_lifi_line']}%), LI.FI fee > 0 on {int(tot_all['lifi_fee_gt_0'])}, integrator fee > 0 on {int(tot_all['integrator_fee_gt_0'])}; "
      f"quoted LI.FI fee {tot_all['lifi_fee_usd']:,.2f} ({tot_all['lifi_bps']} bps of headline USD), integrator fee {tot_all['integrator_fee_usd']:,.2f} ({tot_all['integrator_bps']} bps), other lines {tot_all['other_fee_usd']:,.2f}")
print(f"  swap rows: {int(tot_sw['events'])} events, {tot_sw['amount_usd']:,.2f} USD, {int(tot_sw['with_lifi_line'])} with a LIFI-named line ({tot_sw['pct_with_lifi_line']}%), LI.FI fee {tot_sw['lifi_fee_usd']:,.2f}, integrator fee {tot_sw['integrator_fee_usd']:,.2f}")
print(f"  bridge-only rows: {int(tot_br['events'])} events, {tot_br['amount_usd']:,.2f} USD, {int(tot_br['with_lifi_line'])} with a LIFI-named line ({tot_br['pct_with_lifi_line']}%), any fee line on {int(tot_br['with_any_fee_line'])}, LI.FI fee {tot_br['lifi_fee_usd']:,.2f}, integrator fee {tot_br['integrator_fee_usd']:,.2f}, other lines {tot_br['other_fee_usd']:,.2f}")
print("\nthe twenty largest transfers at >= $100k, swap and bridge-only (headline USD, status, fee line names, LI.FI and integrator fee USD):")
big = run_sql("""WITH rec AS (
        SELECT chain, tx_hash, log_index, 'extension' AS record_from, status, substatus, CAST(quote_fee_names AS VARCHAR) AS quote_fee_names, fee_split_lifi_usd, fee_split_integrator_usd, quote_fee_other_usd, reused_from FROM ref_lifi_api_transfer_status_extension WHERE in_census
        UNION ALL
        SELECT chain, tx_hash, log_index, 'bridge_census', status, substatus, CAST(quote_fee_names AS VARCHAR), fee_split_lifi_usd, fee_split_integrator_usd, quote_fee_other_usd, CAST(reused_from AS VARCHAR) FROM ref_lifi_api_transfer_status_bridge_census)
    SELECT p.chain, p.block_time, coalesce(p.integrator, '') AS integrator, p.bridge, p.symbol, p.has_source_swaps, round(p.amount_usd, 2) AS amount_usd, r.record_from, r.status, r.substatus, r.quote_fee_names,
           round(r.fee_split_lifi_usd, 2) AS lifi_fee_usd, round(r.fee_split_integrator_usd, 2) AS integrator_fee_usd, round(r.quote_fee_other_usd, 2) AS other_fee_usd, r.reused_from
    FROM transfers_priced p LEFT JOIN rec r USING (chain, tx_hash, log_index)
    WHERE p.amount_usd >= 100000 ORDER BY p.amount_usd DESC LIMIT 20""")
display(big)

events  with_record  record_from_extension  record_from_bridge_census  record_from_edge_cases  without_record  http_200  completed  done_not_completed  not_done  with_quote  with_any_fee_line  \
route_kind  usd_band                                                                                                                                                                                                      
all         all            382          382                     73                        309                       0               0       382        381                   1         0         382                203   
            $100k-250k     235          235                     52                        183                       0               0       235        235                   0         0         235                131   
            $250k-1M       125          125                     15                        110                       0               0       125        124                   1         0         125                 61   
            $1M+            22           22                      6                         16                       0               0        22         22                   0         0          22                 11   
swap        all             73           73                     73                          0                       0               0        73         73                   0         0          73                 70   
            $100k-250k      52           52                     52                          0                       0               0        52         52                   0         0          52                 50   
            $250k-1M        15           15                     15                          0                       0               0        15         15                   0         0          15                 14   
            $1M+             6            6                      6                          0                       0               0         6          6                   0         0           6                  6   
bridge_only all            309          309                      0                        309                       0               0       309        308                   1         0         309                133   
            $100k-250k     183          183                      0                        183                       0               0       183        183                   0         0         183                 81   
            $250k-1M       110          110                      0                        110                       0               0       110        109                   1         0         110                 47   
            $1M+            16           16                      0                         16                       0               0        16         16                   0         0          16                  5   

                        with_lifi_line  pct_with_lifi_line  lifi_fee_gt_0  integrator_fee_gt_0  with_batch_status    amount_usd  pct_of_file_usd   sending_usd  lifi_fee_usd  integrator_fee_usd  other_fee_usd  lifi_bps  \
route_kind  usd_band                                                                                                                                                                                                        
all         all                     32                8.38             22                   24                  0  1.684973e+08            54.71  1.684819e+08       2912.58             8858.90       18677.72     0.173   
            $100k-250k              21                8.94             14                   14                  0  3.519275e+07            11.43  3.520208e+07       1702.57             6057.64        4726.68     0.484   
            $250k-1M                 8                6.40              5

every transfer at >= $100k: 382 events, 168,497,251.97 USD (54.71% of the file); with a status record 382 (extension 73, bridge census 309, edge cases 0), without 0; completed 381; with a LIFI-named line 32 (8.38%), LI.FI fee > 0 on 22, integrator fee > 0 on 24; quoted LI.FI fee 2,912.58 (0.173 bps of headline USD), integrator fee 8,858.90 (0.526 bps), other lines 18,677.72
  swap rows: 73 events, 23,034,116.34 USD, 32 with a LIFI-named line (43.84%), LI.FI fee 2,912.58, integrator fee 8,858.90
  bridge-only rows: 309 events, 145,463,135.63 USD, 0 with a LIFI-named line (0.0%), any fee line on 133, LI.FI fee 0.00, integrator fee 0.00, other lines 9,585.41

the twenty largest transfers at >= $100k, swap and bridge-only (headline USD, status, fee line names, LI.FI and integrator fee USD):


,chain,block_time,integrator,bridge,symbol,has_source_swaps,amount_usd,record_from,status,substatus,quote_fee_names,lifi_fee_usd,integrator_fee_usd,other_fee_usd,reused_from
0,ethereum,2026-10-18 16:19:23,_arbitrum,polymerStandard,USDC,False,10002421.67,bridge_census,DONE,COMPLETED,NaN,NaN,NaN,NaN,lifi_api_transfer_status_edge_cases_raw.jsonl
1,ethereum,2026-10-18 16:40:35,_arbitrum,polymerStandard,USDC,False,10002421.67,bridge_census,DONE,COMPLETED,NaN,NaN,NaN,NaN,lifi_api_transfer_status_edge_cases_raw.jsonl
2,ethereum,2026-10-16 13:39:59,Pharos-Bridge,polymerStandard,USDC,False,10000569.17,bridge_census,DONE,COMPLETED,NaN,NaN,NaN,NaN,lifi_api_transfer_status_edge_cases_raw.jsonl
3,ethereum,2026-10-18 17:01:59,_arbitrum,polymerStandard,USDC,False,9521501.02,bridge_census,DONE,COMPLETED,NaN,NaN,NaN,NaN,lifi_api_transfer_status_edge_cases_raw.jsonl
4,arbitrum,2026-10-19 17:57:04,_arbitrum,polymerStandard,USDC,False,9520869.56,bridge_census,DONE,COMPLETED,NaN,NaN,NaN,NaN,lifi_api_transfer_status_edge_cases_raw.jsonl
5,ethereum,2026-10-16 13:54:59,Pharos-Bridge,polymerStandard,USDC,False,6505356.54,bridge_census,DONE,COMPLETED,NaN,NaN,NaN,NaN,lifi_api_transfer_status_edge_cases_raw.jsonl
6,arbitrum,2026-10-15 00:04:58,jumper.exchange,glacis,PYUSD,False,2013311.27,bridge_census,DONE,COMPLETED,Bridge Fee,NaN,NaN,1.60,lifi_api_transfer_status_edge_cases_raw.jsonl
7,ethereum,2026-10-14 00:23:47,_binancewallet,polymerStandard,USDC,False,2000378.17,bridge_census,DONE,COMPLETED,NaN,NaN,NaN,NaN,lifi_api_transfer_status_edge_cases_raw.jsonl
8,arbitrum,2026-10-15 00:53:01,USDai,glacis,PYUSD,True,1655006.40,extension,DONE,COMPLETED,Bridge Fee,NaN,NaN,1.57,lifi_api_transfer_status_edge_cases_raw.jsonl
9,ethereum,2026-10-16 15:52:23,safe-app,glacis,USDG,True,1517757.14,extension,DONE,COMPLETED,LIFI Shared Fee|Bridge Fee,531.32,986.73,0.52,lifi_api_transfer_status_edge_cases_raw.jsonl


### 8.5 Snapshots written by this section, the manifest, and the key check

In [19]:
BRIDGE_FILES = [BRIDGE_CSV, BRIDGE_JSONL.name]
m = manifest_read()
display(pd.DataFrame.from_dict({k: m[k] for k in BRIDGE_FILES}, orient="index")[["fetched_at_utc", "rows", "notebook_section", "source_url"]])
for k in BRIDGE_FILES:
    print(f"{k}: {m[k].get('note')}")
counts_on_disk = {k: (len(pd.read_csv(REF / k, dtype=str)) if k.endswith(".csv") else sum(1 for _ in (REF / k).open())) for k in BRIDGE_FILES}
sizes_mb = {k: round(os.path.getsize(REF / k) / 1e6, 1) for k in BRIDGE_FILES}
print("rows on disk:", counts_on_disk, " manifest agrees:", all(counts_on_disk[k] == m[k]["rows"] for k in BRIDGE_FILES))
print("file sizes (MB):", sizes_mb, " the raw file under 50 MB:", sizes_mb[BRIDGE_JSONL.name] < 50)
print("the committed files this section reads but never writes:", [PRIOR_2600, PRIOR_EDGE, EXT_CSV] + [p.name for p in NEW_JSONLS])
if LIFI_KEY:
    leaked = [k for k in BRIDGE_FILES if LIFI_KEY in (REF / k).read_text()]
    print("files under data/reference containing the key value:", leaked or "none")
    assert not leaked
print(f"calls this run: {RATE['calls']}, HTTP 429: {RATE['n429']}, network errors: {RATE['network_errors']}")

,fetched_at_utc,rows,notebook_section,source_url
lifi_api_transfer_status_bridge_census.csv,2026-11-01T13:56:43+00:00,309,07 §8.2,https://li.quest/v1/status?txHash=<tx_hash>
lifi_api_transfer_status_extension_bridge_census_raw.jsonl,2026-11-01T13:56:43+00:00,284,07 §8.2,https://li.quest/v1/status?txHash=<tx_hash>


lifi_api_transfer_status_bridge_census.csv: third design: census_bridge_100k = every row without has_source_swaps with headline amount_usd >= 100,000 (309 events, 145,463,135.63 USD, 47.2% of the file's headline USD); no draw; 309 rows, one per event, the 77 columns of lifi_api_transfer_status_extension.csv in its order (sample_design = census_bridge_100k, in_census and in_strata false, stratum_* empty); 25 rows reuse the committed record of their hash ({'lifi_api_transfer_status_edge_cases_raw.jsonl': 15, 'lifi_api_transfer_status_sample_2600_raw.jsonl': 10}) with its original fetch time; the rest from 284 new calls: 284 calls from 2026-11-01T13:56:43+00:00 to 2026-11-01T13:59:30+00:00 (102 per minute observed, 0.1 s sleep after each call and a wait until ratelimit-reset whenever ratelimit-remaining fell below 3: 2 such waits); final HTTP statuses {200: 284}; HTTP 429 responses retried before a final response 0 (recorded per call as retries_429); minimum ratelimit-remaining 2 of 100; 